# Original fused NF4 MMA — MICROBENCH_ONLY

No training, model load, optimizer, task score, ownership-share or admission test. This notebook runs source-pinned native bitsandbytes versus one original single-warp16x8 inline-MMA fallback on synthetic q/v-shaped FP16 inputs. The native dispatcher and native MatMul4Bit backward remain unchanged. No full-task installation entrypoint is provided.

Pinned Triton3.6 no longer lowers plain tl.dot to SM75 TensorCores; this original register-fragment mapping uses the public PTX m16n8k8 instruction. It is deliberately small and untuned. CPU fragment/packing proofs and local Triton3.9 offline SM75 compilation are NOT target validation or evidence of speed.

Target preflight must first pass exact W16 decoding, physical lane-layout/MMA/no-FMA codegen gates, fixed engineering forward tolerances, nonfinite classes, native input/bias gradients under fixed cotangents, and the FP32 bias-before-half-rounding witness. Preflight is isolated from every measured cache. Each GPU child has an unchanged80second wall cap; service caps remain authoritative.

Six synthetic M/N pairs use M65,532,2048 and N256,1536; K1536, nested NF4 block64/superblock256, FP16 bias. M532/2048 are task-compatible shapes, not captured task activations. Exact tensor types, contiguous layout, no lazy neg/conj, native no-grad forward scope and T4SM75 are required; other calls fall back unchanged.

Native then candidate uses fresh processes and private empty caches. Cold synchronized wrapper wall includes necessary JIT, allocation, dispatch and GPU work, not pure compilation. Warm CUDA-event intervals include Python launch gaps. No task-speed or share inference. No downloads, installs, credentials or automatic retries. Payload/source/runtime, lock, guardian cleanup, quarantine and idle gates fail closed. Parent alone may submit; incomplete output is censored, never success.


In [ ]:
import os,base64,hashlib,tempfile
from pathlib import Path
assert os.environ.get('SOUP_REPO') and os.environ.get('SOUP_RESULTS')
assert Path('/kaggle/working').is_dir()
PAYLOAD = {'solution/__init__.py': 'IiIiUmVzZWFyY2gtb25seSBndWFyZGVkIGZvcndhcmQgaGVscGVyLiBObyB0YXNrIGluc3RhbGxhdGlvbiBlbnRyeXBvaW50LiIiIgo=', 'solution/fixtures.py': 'IiIiU21hbGwgc3ludGhldGljIHEvdi1zaGFwZWQgaW5wdXRzLCBub3QgY2FwdHVyZWQgdGFzayBhY3RpdmF0aW9ucy4iIiIKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBqc29uCmltcG9ydCB0b3JjaAppbXBvcnQgYml0c2FuZGJ5dGVzIGFzIGJuYgpmcm9tIGJpdHNhbmRieXRlcyBpbXBvcnQgZnVuY3Rpb25hbCBhcyBGCgpFWFBFQ1RFRD1qc29uLmxvYWRzKChQYXRoKF9fZmlsZV9fKS53aXRoX25hbWUoJ25hdGl2ZV9zb3VyY2VfbWFuaWZlc3QuanNvbicpKS5yZWFkX3RleHQoKSkKU0hBUEVTPSgoNjUsMjU2KSwoNTMyLDI1NiksKDIwNDgsMjU2KSwoNjUsMTUzNiksKDUzMiwxNTM2KSwoMjA0OCwxNTM2KSkKCmRlZiBmaXh0dXJlKG0sbix3aXRoX2JpYXM9VHJ1ZSk6CiAgICB0b3JjaC5tYW51YWxfc2VlZCgxNzI5K24pCiAgICB3PXRvcmNoLnJhbmRuKG4sMTUzNixkZXZpY2U9J2N1ZGEnLGR0eXBlPXRvcmNoLmZsb2F0MTYpKi4xCiAgICBwYWNrZWQsc3RhdGU9Ri5xdWFudGl6ZV80Yml0KHcsYmxvY2tzaXplPTY0LGNvbXByZXNzX3N0YXRpc3RpY3M9VHJ1ZSxxdWFudF90eXBlPSduZjQnKQogICAgYT10b3JjaC5yYW5kbihtLDE1MzYsZGV2aWNlPSdjdWRhJyxkdHlwZT10b3JjaC5mbG9hdDE2KSouMQogICAgYmlhcz10b3JjaC5yYW5kbihuLGRldmljZT0nY3VkYScsZHR5cGU9dG9yY2guZmxvYXQxNikqLjAzIGlmIHdpdGhfYmlhcyBlbHNlIE5vbmUKICAgIHJldHVybiBhLHBhY2tlZCxzdGF0ZSxiaWFzCgpkZWYgY2FsbChkYXRhKToKICAgIGEscGFja2VkLHN0YXRlLGJpYXM9ZGF0YQogICAgcmV0dXJuIGJuYi5tYXRtdWxfNGJpdChhLHBhY2tlZC50KCkscXVhbnRfc3RhdGU9c3RhdGUsYmlhcz1iaWFzKQoKZGVmIGNsYXNzZXMoYSxiKToKICAgIGlmIGEuc2hhcGUhPWIuc2hhcGUgb3IgYS5kdHlwZSE9Yi5kdHlwZTpyYWlzZSBSdW50aW1lRXJyb3IoJ1NoYXBlIG9yIGR0eXBlIG1pc21hdGNoJykKICAgIGZvciBmbiBpbiAodG9yY2guaXNuYW4sdG9yY2guaXNwb3NpbmYsdG9yY2guaXNuZWdpbmYpOgogICAgICAgIGlmIG5vdCB0b3JjaC5lcXVhbChmbihhKSxmbihiKSk6cmFpc2UgUnVudGltZUVycm9yKCdOb25maW5pdGUgY2xhc3NlcyBkaWZmZXInKQoKZGVmIGV4YWN0KGEsYik6CiAgICBjbGFzc2VzKGEsYikKICAgIGZpbml0ZT10b3JjaC5pc2Zpbml0ZShhKSZ0b3JjaC5pc2Zpbml0ZShiKQogICAgaWYgbm90IHRvcmNoLmVxdWFsKGFbZmluaXRlXSxiW2Zpbml0ZV0pOnJhaXNlIFJ1bnRpbWVFcnJvcignRmluaXRlIHZhbHVlcyBkaWZmZXInKQogICAgaWYgbm90IHRvcmNoLmVxdWFsKHRvcmNoLnNpZ25iaXQoYVthPT0wXSksdG9yY2guc2lnbmJpdChiW2I9PTBdKSk6cmFpc2UgUnVudGltZUVycm9yKCdTaWduZWQgemVyb3MgZGlmZmVyJykKCmRlZiBlbmdpbmVlcmluZ19jbG9zZShhLGIpOgogICAgY2xhc3NlcyhhLGIpCiAgICBmaW5pdGU9dG9yY2guaXNmaW5pdGUoYSkmdG9yY2guaXNmaW5pdGUoYikKICAgIHRvcmNoLnRlc3RpbmcuYXNzZXJ0X2Nsb3NlKGFbZmluaXRlXSxiW2Zpbml0ZV0sYXRvbD0uMDAzLHJ0b2w9LjAwMikKICAgIGRlbHRhPShhW2Zpbml0ZV0uZmxvYXQoKS1iW2Zpbml0ZV0uZmxvYXQoKSkuYWJzKCkKICAgIHJldHVybiB7J21heF9hYnNfZXJyb3InOmZsb2F0KGRlbHRhLm1heCgpLml0ZW0oKSkgaWYgZGVsdGEubnVtZWwoKSBlbHNlIDAuLAogICAgICAgICAgICAnYXRvbCc6LjAwMywncnRvbCc6LjAwMiwnY3JpdGVyaW9uJzonZml4ZWQgc3ludGhldGljIGVuZ2luZWVyaW5nIGdhdGUsIG5vdCB0YXNrIHF1YWxpdHknfQo=', 'solution/guarded_forward.py': 'IiIiUmVzZWFyY2gtb25seSBmYWxsYmFjayByZXBsYWNlbWVudDsgbmF0aXZlIGRpc3BhdGNoZXIvYXV0b2dyYWQgc3RheSBvd25lZC4KCk5vIHBlcnNpc3RlbnQgaW5zdGFsbGF0aW9uIGFuZCBubyBjdXN0b20gYmFja3dhcmQuIERpcmVjdCBncmFkLWVuYWJsZWQgdXNlIGlzCmluZWxpZ2libGU6IG9ubHkgbmF0aXZlIE1hdE11bDRCaXQuZm9yd2FyZCdzIG5vLWdyYWQgY29udGV4dCBtYXkgcm91dGUgdHJhaW5hYmxlCmlucHV0cyBoZXJlLiBUaGlzIGhlbHBlciBpcyBub3QgYW4gYWRtaXNzaW9uIG9yIGZ1bGwtdGFzayBjYW5kaWRhdGUuCiIiIgpmcm9tIGNvbnRleHRsaWIgaW1wb3J0IGNvbnRleHRtYW5hZ2VyCmltcG9ydCBoYXNobGliCmltcG9ydCBpbnNwZWN0CmltcG9ydCBtYXRoCmltcG9ydCB0b3JjaAppbXBvcnQgdHJpdG9uCmZyb20gLmtlcm5lbCBpbXBvcnQgc291cF9uZjRfbW1hX2xhbmVwcm9iZQoKQ09VTlRTID0geyJjdXN0b20iOiAwLCAibmF0aXZlIjogMH0KQ09NUElMRUQgPSB7fQpGQUxMQkFDS19SRUFTT05TID0ge30KCgpkZWYgcmVhc29uKEEsIEIsIHNoYXBlQiwgYWJzbWF4LCBibG9ja3NpemUsIHF1YW50X3R5cGUsIGJpYXM9Tm9uZSwKICAgICAgICAgICBhYnNtYXhfOGJpdD1Ob25lLCBhYnNtYXhfY29kZT1Ob25lLCBhYnNtYXhfb2Zmc2V0PU5vbmUpOgogICAgaWYgdHlwZShibG9ja3NpemUpIGlzIG5vdCBpbnQgb3IgYmxvY2tzaXplICE9IDY0IG9yIHF1YW50X3R5cGUgIT0gIm5mNCI6CiAgICAgICAgcmV0dXJuICJmb3JtYXQiCiAgICBpZiB0eXBlKHNoYXBlQikgbm90IGluIChsaXN0LCB0dXBsZSwgdG9yY2guU2l6ZSkgb3IgbGVuKHNoYXBlQikgIT0gMjoKICAgICAgICByZXR1cm4gInNoYXBlX21ldGFkYXRhIgogICAgaWYgYW55KHR5cGUoeCkgaXMgbm90IGludCBmb3IgeCBpbiBzaGFwZUIpIG9yIHR1cGxlKHNoYXBlQikgbm90IGluICgoMTUzNiwxNTM2KSwoMjU2LDE1MzYpKToKICAgICAgICByZXR1cm4gInNoYXBlX3Njb3BlIgogICAgbixrID0gc2hhcGVCCiAgICBvcGVyYW5kcyA9IChBLEIsYWJzbWF4LGFic21heF84Yml0LGFic21heF9jb2RlLGFic21heF9vZmZzZXQpCiAgICBpZiBhbnkodHlwZSh4KSBpcyBub3QgdG9yY2guVGVuc29yIGZvciB4IGluIG9wZXJhbmRzKSBvciAoYmlhcyBpcyBub3QgTm9uZSBhbmQgdHlwZShiaWFzKSBpcyBub3QgdG9yY2guVGVuc29yKToKICAgICAgICByZXR1cm4gInRlbnNvcl90eXBlIgogICAgaWYgQS5uZGltIG5vdCBpbiAoMiwzKSBvciBBLnNoYXBlWy0xXSAhPSBrIG9yIChBLm5kaW0gPT0gMyBhbmQgQS5zaGFwZVswXSAhPSA0KToKICAgICAgICByZXR1cm4gImFjdGl2YXRpb25fZ2VvbWV0cnkiCiAgICBtID0gbWF0aC5wcm9kKEEuc2hhcGVbOi0xXSkKICAgIGlmIG5vdCA2NSA8PSBtIDw9IDIwNDg6CiAgICAgICAgcmV0dXJuICJyb3dfc2NvcGUiCiAgICBhbGxfdGVuc29ycyA9IG9wZXJhbmRzICsgKCgpIGlmIGJpYXMgaXMgTm9uZSBlbHNlIChiaWFzLCkpCiAgICBpZiBhbnkoeC5pc19uZWcoKSBvciB4LmlzX2NvbmooKSBvciBub3QgeC5pc19jb250aWd1b3VzKCkgZm9yIHggaW4gYWxsX3RlbnNvcnMpOgogICAgICAgIHJldHVybiAibGF6eV9vcl9zdHJpZGVkIgogICAgaWYgYW55KHguZGV2aWNlICE9IEEuZGV2aWNlIGZvciB4IGluIGFsbF90ZW5zb3JzKSBvciBBLmRldmljZS50eXBlICE9ICJjdWRhIjoKICAgICAgICByZXR1cm4gImRldmljZSIKICAgIGlmIEEuZHR5cGUgIT0gdG9yY2guZmxvYXQxNiBvciBCLmR0eXBlICE9IHRvcmNoLnVpbnQ4IG9yIEIubnVtZWwoKSAhPSBuKmsvLzIgb3IgQi5uZGltIG5vdCBpbiAoMSwyKToKICAgICAgICByZXR1cm4gImFjdGl2YXRpb25fb3JfcGFja2VkX2R0eXBlX2dlb21ldHJ5IgogICAgYmxvY2tzPW4qay8vNjQKICAgIGlmIChhYnNtYXhfOGJpdC5kdHlwZSAhPSB0b3JjaC51aW50OCBvciBhYnNtYXhfOGJpdC5uZGltICE9IDEgb3IgYWJzbWF4XzhiaXQubnVtZWwoKSAhPSBibG9ja3MKICAgICAgICAgICAgb3IgYWJzbWF4LmR0eXBlICE9IHRvcmNoLmZsb2F0MzIgb3IgYWJzbWF4Lm5kaW0gIT0gMSBvciBhYnNtYXgubnVtZWwoKSAhPSB0cml0b24uY2RpdihibG9ja3MsMjU2KQogICAgICAgICAgICBvciBhYnNtYXhfY29kZS5kdHlwZSAhPSB0b3JjaC5mbG9hdDMyIG9yIGFic21heF9jb2RlLm5kaW0gIT0gMSBvciBhYnNtYXhfY29kZS5udW1lbCgpICE9IDI1NgogICAgICAgICAgICBvciBhYnNtYXhfb2Zmc2V0LmR0eXBlICE9IHRvcmNoLmZsb2F0MzIgb3IgYWJzbWF4X29mZnNldC5udW1lbCgpICE9IDEpOgogICAgICAgIHJldHVybiAibmVzdGVkX3NjYWxlX2dlb21ldHJ5IgogICAgaWYgYW55KHgucmVxdWlyZXNfZ3JhZCBmb3IgeCBpbiBvcGVyYW5kc1sxOl0pOgogICAgICAgIHJldHVybiAiZGlmZmVyZW50aWFibGVfbWV0YWRhdGEiCiAgICBpZiBiaWFzIGlzIG5vdCBOb25lIGFuZCAoYmlhcy5kdHlwZSAhPSB0b3JjaC5mbG9hdDE2IG9yIGJpYXMuc2hhcGUgIT0gKG4sKSk6CiAgICAgICAgcmV0dXJuICJiaWFzX2dlb21ldHJ5IgogICAgaWYgdG9yY2guaXNfZ3JhZF9lbmFibGVkKCkgYW5kIChBLnJlcXVpcmVzX2dyYWQgb3IgKGJpYXMgaXMgbm90IE5vbmUgYW5kIGJpYXMucmVxdWlyZXNfZ3JhZCkpOgogICAgICAgIHJldHVybiAiZGlyZWN0X2F1dG9ncmFkX3VzZSIKICAgIHByb3A9dG9yY2guY3VkYS5nZXRfZGV2aWNlX3Byb3BlcnRpZXMoQS5kZXZpY2UpCiAgICBpZiAocHJvcC5tYWpvcixwcm9wLm1pbm9yLHByb3AubXVsdGlfcHJvY2Vzc29yX2NvdW50KSAhPSAoNyw1LDQwKToKICAgICAgICByZXR1cm4gImFyY2hpdGVjdHVyZSIKICAgIHJldHVybiBOb25lCgoKZGVmIG1ha2VfcmVwbGFjZW1lbnQobmF0aXZlKToKICAgIGRlZiByZXBsYWNlbWVudChBLEIsc2hhcGVCLGFic21heCxibG9ja3NpemUscXVhbnRfdHlwZSxiaWFzPU5vbmUsCiAgICAgICAgICAgICAgICAgICAgYWJzbWF4XzhiaXQ9Tm9uZSxhYnNtYXhfY29kZT1Ob25lLGFic21heF9vZmZzZXQ9Tm9uZSk6CiAgICAgICAgYXJncz0oQSxCLHNoYXBlQixhYnNtYXgsYmxvY2tzaXplLHF1YW50X3R5cGUsYmlhcyxhYnNtYXhfOGJpdCxhYnNtYXhfY29kZSxhYnNtYXhfb2Zmc2V0KQogICAgICAgIHdoeT1yZWFzb24oKmFyZ3MpCiAgICAgICAgaWYgd2h5IGlzIG5vdCBOb25lOgogICAgICAgICAgICBDT1VOVFNbIm5hdGl2ZSJdKz0xCiAgICAgICAgICAgIEZBTExCQUNLX1JFQVNPTlNbd2h5XT1GQUxMQkFDS19SRUFTT05TLmdldCh3aHksMCkrMQogICAgICAgICAgICByZXR1cm4gbmF0aXZlKCphcmdzKQogICAgICAgIG4saz1zaGFwZUI7bT1BLm51bWVsKCkvL2sKICAgICAgICBvdXQ9dG9yY2guZW1wdHkoKCpBLnNoYXBlWzotMV0sbiksZGV2aWNlPUEuZGV2aWNlLGR0eXBlPXRvcmNoLmZsb2F0MTYpCiAgICAgICAgY29tcGlsZWQ9c291cF9uZjRfbW1hX2xhbmVwcm9iZVsodHJpdG9uLmNkaXYobSwxNiksdHJpdG9uLmNkaXYobiw4KSldKAogICAgICAgICAgICBBLEIsYWJzbWF4XzhiaXQsYWJzbWF4X2NvZGUsYWJzbWF4LGFic21heF9vZmZzZXQsQSBpZiBiaWFzIGlzIE5vbmUgZWxzZSBiaWFzLG91dCwKICAgICAgICAgICAgbSxuLGssYmlhcyBpcyBub3QgTm9uZSxudW1fd2FycHM9MSxudW1fc3RhZ2VzPTEsZW5hYmxlX2ZwX2Z1c2lvbj1GYWxzZSkKICAgICAgICBDT01QSUxFRFsobixrLGJpYXMgaXMgbm90IE5vbmUpXT1jb21waWxlZAogICAgICAgIENPVU5UU1siY3VzdG9tIl0rPTEKICAgICAgICByZXR1cm4gb3V0CiAgICByZXR1cm4gcmVwbGFjZW1lbnQKCgpkZWYgdmVyaWZ5X25hdGl2ZV9vd25lcihrZXksdmFsdWUsZXhwZWN0ZWQpOgogICAgb3duZXJzPXsnZmFsbGJhY2snOignYml0c2FuZGJ5dGVzLmJhY2tlbmRzLmN1ZGEub3BzJywnX2RlcXVhbnRfbGluZWFyX2ZhbGxiYWNrJyksCiAgICAgICAgICAgICdiYWNrd2FyZCc6KCdiaXRzYW5kYnl0ZXMuYXV0b2dyYWQuX2Z1bmN0aW9ucycsJ01hdE11bDRCaXQuYmFja3dhcmQnKX0KICAgIGlmIChub3QgaW5zcGVjdC5pc2Z1bmN0aW9uKHZhbHVlKSBvciBoYXNhdHRyKHZhbHVlLCdfX3dyYXBwZWRfXycpCiAgICAgICAgICAgIG9yICh2YWx1ZS5fX21vZHVsZV9fLHZhbHVlLl9fcXVhbG5hbWVfXykhPW93bmVyc1trZXldKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0ZvcmVpZ24gbmF0aXZlIGNhbGxhYmxlIG93bmVyOiAnK2tleSkKICAgIGlmIGhhc2hsaWIuc2hhMjU2KGluc3BlY3QuZ2V0c291cmNlKHZhbHVlKS5lbmNvZGUoKSkuaGV4ZGlnZXN0KCkhPWV4cGVjdGVkW2tleV06CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdOYXRpdmUgb3duZXJzaGlwIHNvdXJjZSBtaXNtYXRjaDogJytrZXkpCgoKQGNvbnRleHRtYW5hZ2VyCmRlZiB0ZW1wb3JhcnlfbmF0aXZlX2ZhbGxiYWNrKGV4cGVjdGVkKToKICAgICIiIk9ubHkgZm9yIGNvbnRyb2xsZWQgbWljcm9iZW5jaCBjaGlsZHJlbjsgZmFpbCBjbG9zZWQgb24gZm9yZWlnbiBvd25lcnMuIiIiCiAgICBpbXBvcnQgYml0c2FuZGJ5dGVzLmJhY2tlbmRzLmN1ZGEub3BzIGFzIG9wcwogICAgZnJvbSBiaXRzYW5kYnl0ZXMuYXV0b2dyYWQuX2Z1bmN0aW9ucyBpbXBvcnQgTWF0TXVsNEJpdAogICAgbmF0aXZlPW9wcy5fZGVxdWFudF9saW5lYXJfZmFsbGJhY2s7YmFja3dhcmQ9TWF0TXVsNEJpdC5iYWNrd2FyZAogICAgZm9yIGtleSx2YWx1ZSBpbiAoKCJmYWxsYmFjayIsbmF0aXZlKSwoImJhY2t3YXJkIixiYWNrd2FyZCkpOgogICAgICAgIHZlcmlmeV9uYXRpdmVfb3duZXIoa2V5LHZhbHVlLGV4cGVjdGVkKQogICAgcmVwbGFjZW1lbnQ9bWFrZV9yZXBsYWNlbWVudChuYXRpdmUpCiAgICBvcHMuX2RlcXVhbnRfbGluZWFyX2ZhbGxiYWNrPXJlcGxhY2VtZW50CiAgICB0cnk6CiAgICAgICAgeWllbGQgcmVwbGFjZW1lbnQKICAgICAgICBpZiBNYXRNdWw0Qml0LmJhY2t3YXJkIGlzIG5vdCBiYWNrd2FyZDoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJOYXRpdmUgYmFja3dhcmQgb3duZXIgY2hhbmdlZCIpCiAgICAgICAgaWYgb3BzLl9kZXF1YW50X2xpbmVhcl9mYWxsYmFjayBpcyBub3QgcmVwbGFjZW1lbnQ6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiRmFsbGJhY2sgb3duZXIgY2hhbmdlZCIpCiAgICBmaW5hbGx5OgogICAgICAgIGlmIG9wcy5fZGVxdWFudF9saW5lYXJfZmFsbGJhY2sgaXMgcmVwbGFjZW1lbnQ6CiAgICAgICAgICAgIG9wcy5fZGVxdWFudF9saW5lYXJfZmFsbGJhY2s9bmF0aXZlCg==', 'solution/kernel.py': 'IiIiT3JpZ2luYWwgc2luZ2xlLXdhcnAgcmVnaXN0ZXItZnJhZ21lbnQgZmVhc2liaWxpdHkgcHJvYmUsIG5vdCBhbiBpbnN0YWxsZXIuCgpJbnN0cnVjdGlvbiBhbmQgZnJhZ21lbnQgbWFwcGluZyBmcm9tIE5WSURJQSBQVFggSVNBIG0xNm44azggZG9jdW1lbnRhdGlvbiwKbm90IGEgY29waWVkIHZlbmRvciBHRU1NLiBQbGFpbiB0bC5kb3Qgbm8gbG9uZ2VyIHNlbGVjdHMgU003NSBNTUEgaW4gVHJpdG9uMy42LgpUYXJnZXQgcHJlZmxpZ2h0IG11c3QgdmVyaWZ5IDFEIGxhbmUgbGF5b3V0IGFuZCBleGFjdCBmcmFnbWVudCBtYXBwaW5nIGJlZm9yZSB1c2UuCiIiIgppbXBvcnQgdHJpdG9uCmltcG9ydCB0cml0b24ubGFuZ3VhZ2UgYXMgdGwKZnJvbSAubmY0X2Zvcm1hdCBpbXBvcnQgX25mNF92YWx1ZQoKCkB0cml0b24uaml0CmRlZiBfcGFja19oYWxmMihhLCBiKToKICAgIHJldHVybiBhLnRvKHRsLnVpbnQxNiwgYml0Y2FzdD1UcnVlKS50byh0bC51aW50MzIpIHwgKGIudG8odGwudWludDE2LCBiaXRjYXN0PVRydWUpLnRvKHRsLnVpbnQzMikgPDwgMTYpCgoKQHRyaXRvbi5qaXQKZGVmIF9tbWExNjg4KGEwLCBhMSwgYjAsIGMwLCBjMSwgYzIsIGMzKToKICAgIHJldHVybiB0bC5pbmxpbmVfYXNtX2VsZW1lbnR3aXNlKAogICAgICAgICJtbWEuc3luYy5hbGlnbmVkLm0xNm44azgucm93LmNvbC5mMzIuZjE2LmYxNi5mMzIgeyQwLCQxLCQyLCQzfSwgeyQ0LCQ1fSwgeyQ2fSwgeyQ3LCQ4LCQ5LCQxMH07IiwKICAgICAgICBjb25zdHJhaW50cz0iPWYsPWYsPWYsPWYscixyLHIsZixmLGYsZiIsCiAgICAgICAgYXJncz1bYTAsYTEsYjAsYzAsYzEsYzIsYzNdLGR0eXBlPSh0bC5mbG9hdDMyLHRsLmZsb2F0MzIsdGwuZmxvYXQzMix0bC5mbG9hdDMyKSwKICAgICAgICBpc19wdXJlPVRydWUscGFjaz0xKQoKCkB0cml0b24uaml0KGRvX25vdF9zcGVjaWFsaXplPVsnTSddKQpkZWYgc291cF9uZjRfbW1hX2xhbmVwcm9iZShBLCBQQUNLRUQsIFFBQlMsIENPREUyLCBBQlMyLCBPRkZTRVQsIEJpYXMsIE91dCwKICAgICAgICAgICAgICAgICAgICAgICAgICBNLCBOOnRsLmNvbnN0ZXhwciwgSzp0bC5jb25zdGV4cHIsIEJJQVM6dGwuY29uc3RleHByKToKICAgIHRsLnN0YXRpY19hc3NlcnQoSyAlIDY0ID09IDApCiAgICAjIE9uZSB3YXJwICsgMzIgc2NhbGFyIGxvZ2ljYWwgZWxlbWVudHMuIENvZGVnZW4gZ2F0ZSByZWplY3RzIGFub3RoZXIgbGF5b3V0LgogICAgbGFuZT10bC5hcmFuZ2UoMCwzMikKICAgIGdyb3VwPWxhbmUvLzQKICAgIHBhaXI9bGFuZSU0CiAgICBtMD10bC5wcm9ncmFtX2lkKDApKjE2K2dyb3VwCiAgICBuMD10bC5wcm9ncmFtX2lkKDEpKjgrZ3JvdXAKICAgIGMwPXRsLmZ1bGwoKDMyLCksMCx0bC5mbG9hdDMyKTtjMT10bC5mdWxsKCgzMiwpLDAsdGwuZmxvYXQzMikKICAgIGMyPXRsLmZ1bGwoKDMyLCksMCx0bC5mbG9hdDMyKTtjMz10bC5mdWxsKCgzMiwpLDAsdGwuZmxvYXQzMikKICAgIGZvciBibG9jayBpbiB0bC5yYW5nZSgwLEsvLzY0LGxvb3BfdW5yb2xsX2ZhY3Rvcj0xKToKICAgICAgICBzY2FsZV9ibG9jaz1uMCooSy8vNjQpK2Jsb2NrCiAgICAgICAgcWk9dGwubG9hZChRQUJTK3NjYWxlX2Jsb2NrLG4wPE4sb3RoZXI9MCkudG8odGwuaW50MzIpCiAgICAgICAgY3Y9dGwubG9hZChDT0RFMitxaSxuMDxOLG90aGVyPTApLnRvKHRsLmZsb2F0MzIpCiAgICAgICAgYWI9dGwubG9hZChBQlMyK3NjYWxlX2Jsb2NrLy8yNTYsbjA8TixvdGhlcj0wKS50byh0bC5mbG9hdDMyKQogICAgICAgIHByb2R1Y3Q9Y3YqYWIKICAgICAgICBzY2FsZT1wcm9kdWN0K3RsLmxvYWQoT0ZGU0VUKS50byh0bC5mbG9hdDMyKQogICAgICAgIGZvciBjaHVuayBpbiB0bC5yYW5nZSgwLDgsbG9vcF91bnJvbGxfZmFjdG9yPTEpOgogICAgICAgICAgICBrMD1ibG9jayo2NCtjaHVuayo4K3BhaXIqMgogICAgICAgICAgICBhMDA9dGwubG9hZChBK20wKksrazAsbTA8TSxvdGhlcj0wKS50byh0bC5mbG9hdDE2KQogICAgICAgICAgICBhMDE9dGwubG9hZChBK20wKksrazArMSxtMDxNLG90aGVyPTApLnRvKHRsLmZsb2F0MTYpCiAgICAgICAgICAgIGExMD10bC5sb2FkKEErKG0wKzgpKksrazAsbTArODxNLG90aGVyPTApLnRvKHRsLmZsb2F0MTYpCiAgICAgICAgICAgIGExMT10bC5sb2FkKEErKG0wKzgpKksrazArMSxtMCs4PE0sb3RoZXI9MCkudG8odGwuZmxvYXQxNikKICAgICAgICAgICAgYnl0ZT10bC5sb2FkKFBBQ0tFRCsobjAqSytrMCkvLzIsbjA8TixvdGhlcj0wKS50byh0bC51aW50MzIpCiAgICAgICAgICAgIGIwPShfbmY0X3ZhbHVlKGJ5dGU+PjQpKnNjYWxlKS50byh0bC5mbG9hdDE2LGZwX2Rvd25jYXN0X3JvdW5kaW5nPSdydG5lJykKICAgICAgICAgICAgYjE9KF9uZjRfdmFsdWUoYnl0ZSYxNSkqc2NhbGUpLnRvKHRsLmZsb2F0MTYsZnBfZG93bmNhc3Rfcm91bmRpbmc9J3J0bmUnKQogICAgICAgICAgICBjMCxjMSxjMixjMz1fbW1hMTY4OChfcGFja19oYWxmMihhMDAsYTAxKSxfcGFja19oYWxmMihhMTAsYTExKSxfcGFja19oYWxmMihiMCxiMSksYzAsYzEsYzIsYzMpCiAgICBjb2w9dGwucHJvZ3JhbV9pZCgxKSo4K3BhaXIqMgogICAgaWYgQklBUzoKICAgICAgICBiaWFzMD10bC5sb2FkKEJpYXMrY29sLGNvbDxOLG90aGVyPTApLnRvKHRsLmZsb2F0MzIpCiAgICAgICAgYmlhczE9dGwubG9hZChCaWFzK2NvbCsxLGNvbCsxPE4sb3RoZXI9MCkudG8odGwuZmxvYXQzMikKICAgICAgICBjMD1jMCtiaWFzMDtjMT1jMStiaWFzMTtjMj1jMitiaWFzMDtjMz1jMytiaWFzMQogICAgdGwuc3RvcmUoT3V0K20wKk4rY29sLGMwLnRvKHRsLmZsb2F0MTYpLCAobTA8TSkmKGNvbDxOKSkKICAgIHRsLnN0b3JlKE91dCttMCpOK2NvbCsxLGMxLnRvKHRsLmZsb2F0MTYpLChtMDxNKSYoY29sKzE8TikpCiAgICB0bC5zdG9yZShPdXQrKG0wKzgpKk4rY29sLGMyLnRvKHRsLmZsb2F0MTYpLChtMCs4PE0pJihjb2w8TikpCiAgICB0bC5zdG9yZShPdXQrKG0wKzgpKk4rY29sKzEsYzMudG8odGwuZmxvYXQxNiksKG0wKzg8TSkmKGNvbCsxPE4pKQo=', 'solution/native_source_manifest.json': 'ewogICJmYWxsYmFjayI6ICJhZDA5Y2U4MWIyNWNiMGI4OTEzZjYxYzQ3MGIxYjc5NDJhYzQ1NGJmYzhhYTI3ZmUzYTVhYTdmYTBkMTVjYzZmIiwKICAiYmFja3dhcmQiOiAiMDUzZTg3MTkzMTlhNTI2YzYxZjI4OGEzNzIyMWJhNTIwMGRlOWVhYTIwZGFlN2M3ZTQ1Mzg2MzE1ZjI4NGNkMiIKfQo=', 'solution/nf4_format.py': 'IiIiUHVibGljIE5GNCBGUDMyIGZvcm1hdCBjb25zdGFudHM7IG9yaWdpbmFsIHNlbGVjdG9yLCBubyB2ZW5kb3Iga2VybmVsLiIiIgppbXBvcnQgdHJpdG9uCmltcG9ydCB0cml0b24ubGFuZ3VhZ2UgYXMgdGwKCkB0cml0b24uaml0CmRlZiBfbmY0X3ZhbHVlKGNvZGUpOgogICAgIyBBbGwgbGl0ZXJhbHMgYXJlIGV4YWN0bHkgdGhlIGNhbm9uaWNhbCBGUDMyIHRhYmxlIHZhbHVlcy4gVHdvIGluZGVwZW5kZW50CiAgICAjIHNlbGVjdGlvbiBjaGFpbnMga2VlcCB0aGUgdGFibGUgdmlzaWJsZSBmb3IgdGhlIHNjYWxhciBzb3VyY2UgdGVzdC4KICAgIG5lZ2F0aXZlID0gdGwud2hlcmUoY29kZSA9PSAwLCAtMS4wLCAtMC42OTYxOTI4MDA5OTg2ODc3KQogICAgbmVnYXRpdmUgPSB0bC53aGVyZShjb2RlID09IDIsIC0wLjUyNTA3MzA1MTQ1MjYzNjcsIG5lZ2F0aXZlKQogICAgbmVnYXRpdmUgPSB0bC53aGVyZShjb2RlID09IDMsIC0wLjM5NDkxNzQ4ODA5ODE0NDUzLCBuZWdhdGl2ZSkKICAgIG5lZ2F0aXZlID0gdGwud2hlcmUoY29kZSA9PSA0LCAtMC4yODQ0NDEzODE2OTI4ODYzNSwgbmVnYXRpdmUpCiAgICBuZWdhdGl2ZSA9IHRsLndoZXJlKGNvZGUgPT0gNSwgLTAuMTg0NzczNDMwMjI4MjMzMzQsIG5lZ2F0aXZlKQogICAgbmVnYXRpdmUgPSB0bC53aGVyZShjb2RlID09IDYsIC0wLjA5MTA1MDAzNjI1MTU0NDk1LCBuZWdhdGl2ZSkKICAgIG5lZ2F0aXZlID0gdGwud2hlcmUoY29kZSA9PSA3LCAwLjAsIG5lZ2F0aXZlKQogICAgcG9zaXRpdmUgPSB0bC53aGVyZShjb2RlID09IDgsIDAuMDc5NTgwMjk5NTU2MjU1MzQsIDAuMTYwOTMwMjAxNDExMjQ3MjUpCiAgICBwb3NpdGl2ZSA9IHRsLndoZXJlKGNvZGUgPT0gMTAsIDAuMjQ2MTEyMzAxOTQ1Njg2MzQsIHBvc2l0aXZlKQogICAgcG9zaXRpdmUgPSB0bC53aGVyZShjb2RlID09IDExLCAwLjMzNzkxNTI0MTcxODI5MjI0LCBwb3NpdGl2ZSkKICAgIHBvc2l0aXZlID0gdGwud2hlcmUoY29kZSA9PSAxMiwgMC40NDA3MDk4MjkzMzA0NDQzNCwgcG9zaXRpdmUpCiAgICBwb3NpdGl2ZSA9IHRsLndoZXJlKGNvZGUgPT0gMTMsIDAuNTYyNjE3MDAzOTE3Njk0MSwgcG9zaXRpdmUpCiAgICBwb3NpdGl2ZSA9IHRsLndoZXJlKGNvZGUgPT0gMTQsIDAuNzIyOTU2ODM2MjIzNjAyMywgcG9zaXRpdmUpCiAgICBwb3NpdGl2ZSA9IHRsLndoZXJlKGNvZGUgPT0gMTUsIDEuMCwgcG9zaXRpdmUpCiAgICByZXR1cm4gdGwud2hlcmUoY29kZSA8IDgsIG5lZ2F0aXZlLCBwb3NpdGl2ZSkK', 'solution/operand_probe.py': 'IiIiUHJlZmxpZ2h0LW9ubHkgZGlyZWN0IFcxNiBvYnNlcnZhdGlvbjsgbm90IHBhcnQgb2YgbWVhc3VyZWQgZm9yd2FyZC4iIiIKaW1wb3J0IHRyaXRvbgppbXBvcnQgdHJpdG9uLmxhbmd1YWdlIGFzIHRsCmZyb20gLm5mNF9mb3JtYXQgaW1wb3J0IF9uZjRfdmFsdWUKCkB0cml0b24uaml0CmRlZiBvcGVyYW5kX3Byb2JlKFBBQ0tFRCxRQUJTLENPREUyLEFCUzIsT0ZGU0VULE9VVCxFOnRsLmNvbnN0ZXhwcixCTE9DSzp0bC5jb25zdGV4cHIpOgogICAgaT10bC5wcm9ncmFtX2lkKDApKkJMT0NLK3RsLmFyYW5nZSgwLEJMT0NLKQogICAgcT10bC5sb2FkKFBBQ0tFRCtpLy8yLGk8RSxvdGhlcj0wKS50byh0bC51aW50MzIpCiAgICBjb2RlPXRsLndoZXJlKChpJjEpPT0wLHE+PjQscSYxNSkKICAgIGJsb2NrPWkvLzY0CiAgICBpbmRleD10bC5sb2FkKFFBQlMrYmxvY2ssaTxFLG90aGVyPTApLnRvKHRsLmludDMyKQogICAgY3Y9dGwubG9hZChDT0RFMitpbmRleCxpPEUsb3RoZXI9MCkudG8odGwuZmxvYXQzMikKICAgIHNjYWxlMj10bC5sb2FkKEFCUzIrYmxvY2svLzI1NixpPEUsb3RoZXI9MCkudG8odGwuZmxvYXQzMikKICAgIHByb2R1Y3Q9Y3Yqc2NhbGUyCiAgICBzY2FsZT1wcm9kdWN0K3RsLmxvYWQoT0ZGU0VUKS50byh0bC5mbG9hdDMyKQogICAgdz0oX25mNF92YWx1ZShjb2RlKSpzY2FsZSkudG8odGwuZmxvYXQxNixmcF9kb3duY2FzdF9yb3VuZGluZz0ncnRuZScpCiAgICB0bC5zdG9yZShPVVQraSx3LGk8RSkK', 'solution/web_preflight.py': 'IiIiVGFyZ2V0LW9ubHkgb3JpZ2luYWwgbGFuZSBNTUEgYW5kIHVuY2hhbmdlZCBuYXRpdmUtYmFja3dhcmQgcHJlZmxpZ2h0LiIiIgppbXBvcnQgYXJncGFyc2UsaGFzaGxpYixqc29uLG9zLHRyYWNlYmFjawpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKClBST0dSRVNTID0ge30KCmRlZiBydW4oKToKICAgIGlmIG9zLmVudmlyb24uZ2V0KCdUUklUT05fSU5URVJQUkVUJyk6cmFpc2UgUnVudGltZUVycm9yKCdJbnRlcnByZXRlciBpcyBub3QgYSBkZXZpY2UgZ2F0ZScpCiAgICBpbXBvcnQgdG9yY2gsdHJpdG9uLGJpdHNhbmRieXRlcyBhcyBibmIKICAgIGZyb20gYml0c2FuZGJ5dGVzIGltcG9ydCBmdW5jdGlvbmFsIGFzIEYKICAgIGZyb20gYml0c2FuZGJ5dGVzLmF1dG9ncmFkLl9mdW5jdGlvbnMgaW1wb3J0IE1hdE11bDRCaXQKICAgIGZyb20gLiBpbXBvcnQgZ3VhcmRlZF9mb3J3YXJkIGFzIGcKICAgIGZyb20gLmZpeHR1cmVzIGltcG9ydCBFWFBFQ1RFRCxTSEFQRVMsZml4dHVyZSxjYWxsLGV4YWN0LGVuZ2luZWVyaW5nX2Nsb3NlCiAgICBmcm9tIC5vcGVyYW5kX3Byb2JlIGltcG9ydCBvcGVyYW5kX3Byb2JlCiAgICBpZiAodG9yY2guX192ZXJzaW9uX18hPScyLjExLjArY3UxMjgnIG9yIHRyaXRvbi5fX3ZlcnNpb25fXyE9JzMuNi4wJwogICAgICAgICAgICBvciBibmIuX192ZXJzaW9uX18hPScwLjUwLjInIG9yIHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCkhPTEKICAgICAgICAgICAgb3IgdG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoMCkhPSdUZXNsYSBUNCcgb3IgdG9yY2guY3VkYS5nZXRfZGV2aWNlX2NhcGFiaWxpdHkoMCkhPSg3LDUpKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0V4YWN0IHRhcmdldCBydW50aW1lIHJlcXVpcmVkJykKICAgIHJvd3M9W107YmFja3dhcmQ9TWF0TXVsNEJpdC5iYWNrd2FyZAogICAgZm9yIG0sbiBpbiBTSEFQRVM6CiAgICAgICAgZm9yIGJpYXMgaW4gKEZhbHNlLFRydWUpOgogICAgICAgICAgICBQUk9HUkVTUy51cGRhdGUoc3RhZ2U9ImZvcndhcmRfb3BlcmFuZCIsY2FzZT17Ik0iOm0sIk4iOm4sImJpYXMiOmJpYXN9LGNvbXBsZXRlZF9mb3J3YXJkX3Jvd3M9bGVuKHJvd3MpKQogICAgICAgICAgICBkYXRhPWZpeHR1cmUobSxuLGJpYXMpO2EscCxzLGI9ZGF0YQogICAgICAgICAgICBleHBlY3RlZD1jYWxsKGRhdGEpCiAgICAgICAgICAgIHc9Ri5kZXF1YW50aXplXzRiaXQocCxzKTtvYnNlcnZlZD10b3JjaC5lbXB0eV9saWtlKHcpCiAgICAgICAgICAgIG9wZXJhbmRfcHJvYmVbKHRyaXRvbi5jZGl2KHcubnVtZWwoKSwyNTYpLCldKHAscy5hYnNtYXgscy5zdGF0ZTIuY29kZSxzLnN0YXRlMi5hYnNtYXgscy5vZmZzZXQsb2JzZXJ2ZWQsdy5udW1lbCgpLDI1NiwKICAgICAgICAgICAgICAgIG51bV93YXJwcz00LGVuYWJsZV9mcF9mdXNpb249RmFsc2UpCiAgICAgICAgICAgIGV4YWN0KG9ic2VydmVkLHcpCiAgICAgICAgICAgIFBST0dSRVNTLnVwZGF0ZShzdGFnZT0iY2FuZGlkYXRlX2ZvcndhcmQiLG9wZXJhbmRfZnAxNl9leGFjdF9mb3JfY3VycmVudF9jYXNlPVRydWUpCiAgICAgICAgICAgIGJlZm9yZT1nLkNPVU5UU1snY3VzdG9tJ10KICAgICAgICAgICAgd2l0aCBnLnRlbXBvcmFyeV9uYXRpdmVfZmFsbGJhY2soRVhQRUNURUQpOmFjdHVhbD1jYWxsKGRhdGEpCiAgICAgICAgICAgIFBST0dSRVNTLnVwZGF0ZShjb3VudHM9ZGljdChnLkNPVU5UUyksZmFsbGJhY2tfcmVhc29ucz1kaWN0KGcuRkFMTEJBQ0tfUkVBU09OUykpCiAgICAgICAgICAgIGlmIGcuQ09VTlRTWydjdXN0b20nXSE9YmVmb3JlKzE6cmFpc2UgUnVudGltZUVycm9yKCdDYW5kaWRhdGUgZmFsbGJhY2sgaW5hY3RpdmU7IHJlYXNvbnM9JytyZXByKGcuRkFMTEJBQ0tfUkVBU09OUykpCiAgICAgICAgICAgIGVycm9yPWVuZ2luZWVyaW5nX2Nsb3NlKGFjdHVhbCxleHBlY3RlZCkKICAgICAgICAgICAgcm93cy5hcHBlbmQoeydNJzptLCdOJzpuLCdLJzoxNTM2LCdiaWFzJzpiaWFzLCdvcGVyYW5kX2ZwMTZfZXhhY3QnOlRydWUsKiplcnJvcn0pCiAgICBQUk9HUkVTUy51cGRhdGUoc3RhZ2U9ImNvZGVnZW4iLGNvbXBsZXRlZF9mb3J3YXJkX3Jvd3M9bGVuKHJvd3MpKQogICAgY29kZWdlbj1bXQogICAgZm9yIGtleSxjIGluIGcuQ09NUElMRUQuaXRlbXMoKToKICAgICAgICBwdHg9Yy5hc21bJ3B0eCddO2lyPWMuYXNtWyd0dGdpciddCiAgICAgICAgbGF5b3V0cz1bbGluZSBmb3IgbGluZSBpbiBpci5zcGxpdGxpbmVzKCkgaWYgJyN0dGcuYmxvY2tlZDwnIGluIGxpbmVdCiAgICAgICAgZXhwZWN0ZWRfbGF5b3V0PSdzaXplUGVyVGhyZWFkID0gWzFdLCB0aHJlYWRzUGVyV2FycCA9IFszMl0sIHdhcnBzUGVyQ1RBID0gWzFdLCBvcmRlciA9IFswXScKICAgICAgICBpZiAoJy50YXJnZXQgc21fNzUnIG5vdCBpbiBwdHggb3IgJ21tYS5zeW5jLmFsaWduZWQubTE2bjhrOC5yb3cuY29sLmYzMi5mMTYuZjE2LmYzMicgbm90IGluIHB0eAogICAgICAgICAgICAgICAgb3IgJ2ZtYS5ybi5mMzInIGluIHB0eCBvciAnLmZ0eicgaW4gcHR4CiAgICAgICAgICAgICAgICBvciBsZW4obGF5b3V0cykhPTEgb3IgZXhwZWN0ZWRfbGF5b3V0IG5vdCBpbiBsYXlvdXRzWzBdKToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdTTTc1IGxhbmUgbGF5b3V0L01NQS9zZXBhcmF0ZS1zY2FsZSBjb2RlZ2VuIGdhdGUgZmFpbGVkJykKICAgICAgICBjb2RlZ2VuLmFwcGVuZCh7J2tleSc6bGlzdChrZXkpLCdwdHhfc2hhMjU2JzpoYXNobGliLnNoYTI1NihwdHguZW5jb2RlKCkpLmhleGRpZ2VzdCgpLAogICAgICAgICAgICAndHRnaXJfc2hhMjU2JzpoYXNobGliLnNoYTI1Nihpci5lbmNvZGUoKSkuaGV4ZGlnZXN0KCksJ21tYSc6VHJ1ZSwnbm9fc2NhbGFyX2ZtYV9vcl9mdHonOlRydWUsJ2xhbmVfbGF5b3V0X3ZlcmlmaWVkJzpUcnVlfSkKICAgIFBST0dSRVNTLnVwZGF0ZShzdGFnZT0ibmF0aXZlX2JhY2t3YXJkIikKICAgIGdyYWRpZW50cz1bXQogICAgZm9yIG4gaW4gKDI1NiwxNTM2KToKICAgICAgICBkYXRhPWZpeHR1cmUoNjUsbixUcnVlKQogICAgICAgIGZvciBzY2FsZSBpbiAoMS4sMTI4Liw2NTUzNi4pOgogICAgICAgICAgICBkZWYgb25lKGN1c3RvbSk6CiAgICAgICAgICAgICAgICBhLHAscyxiPWRhdGE7YT1hLmRldGFjaCgpLmNsb25lKCkucmVxdWlyZXNfZ3JhZF8oKTtiPWIuZGV0YWNoKCkuY2xvbmUoKS5yZXF1aXJlc19ncmFkXygpCiAgICAgICAgICAgICAgICBjdXJyZW50PShhLHAscyxiKQogICAgICAgICAgICAgICAgaWYgY3VzdG9tOgogICAgICAgICAgICAgICAgICAgIHdpdGggZy50ZW1wb3JhcnlfbmF0aXZlX2ZhbGxiYWNrKEVYUEVDVEVEKTpvdXQ9Y2FsbChjdXJyZW50KQogICAgICAgICAgICAgICAgZWxzZTpvdXQ9Y2FsbChjdXJyZW50KQogICAgICAgICAgICAgICAgIyBGaXhlZCBjb3RhbmdlbnQgaW5kZXBlbmRlbnQgb2YgZWl0aGVyIG51bWVyaWNhbCBmb3J3YXJkIG91dHB1dC4KICAgICAgICAgICAgICAgIHVwPXRvcmNoLmxpbnNwYWNlKC0uMDEsLjAxLG91dC5udW1lbCgpLGRldmljZT0nY3VkYScpLnJlc2hhcGVfYXMob3V0KS50byhvdXQuZHR5cGUpKnNjYWxlCiAgICAgICAgICAgICAgICBnYSxnYj10b3JjaC5hdXRvZ3JhZC5ncmFkKG91dCwoYSxiKSx1cCkKICAgICAgICAgICAgICAgIHJldHVybiBnYSxnYgogICAgICAgICAgICByZWZlcmVuY2U9b25lKEZhbHNlKTtiZWZvcmU9Zy5DT1VOVFNbJ2N1c3RvbSddO2FjdHVhbD1vbmUoVHJ1ZSkKICAgICAgICAgICAgaWYgZy5DT1VOVFNbJ2N1c3RvbSddIT1iZWZvcmUrMTpyYWlzZSBSdW50aW1lRXJyb3IoJ05hdGl2ZSBhdXRvZ3JhZCBmb3J3YXJkIGJ5cGFzc2VkIGNhbmRpZGF0ZScpCiAgICAgICAgICAgIGZvciB4LHkgaW4gemlwKGFjdHVhbCxyZWZlcmVuY2UpOmV4YWN0KHgseSkKICAgICAgICAgICAgZ3JhZGllbnRzLmFwcGVuZCh7J04nOm4sJ3NjYWxlJzpzY2FsZSwnbmF0aXZlX2lucHV0X2FuZF9iaWFzX2dyYWRpZW50c19leGFjdCc6VHJ1ZX0pCiAgICAjIE5vbmZpbml0ZS96ZXJvLWNsYXNzIGNoZWNrcyBjYW5ub3QgYmUgcmVwbGFjZWQgYnkgdG9sZXJhbmNlcy4KICAgIFBST0dSRVNTLnVwZGF0ZShzdGFnZT0iY2xhc3Nlc19hbmRfYmlhc193aXRuZXNzIikKICAgIGNsYXNzZXM9W10KICAgIGZvciBtb2RlIGluICgnemVybycsJ25hbicsJ3Bvc2luZicsJ25lZ2luZicpOgogICAgICAgIGRhdGE9bGlzdChmaXh0dXJlKDY1LDI1NixGYWxzZSkpO2RhdGFbMF0uemVyb18oKQogICAgICAgIGlmIG1vZGUhPSd6ZXJvJzpkYXRhWzBdWzosMF09ZGljdChuYW49ZmxvYXQoJ25hbicpLHBvc2luZj1mbG9hdCgnaW5mJyksbmVnaW5mPS1mbG9hdCgnaW5mJykpW21vZGVdCiAgICAgICAgZXhwZWN0ZWQ9Y2FsbChkYXRhKQogICAgICAgIHdpdGggZy50ZW1wb3JhcnlfbmF0aXZlX2ZhbGxiYWNrKEVYUEVDVEVEKTphY3R1YWw9Y2FsbChkYXRhKQogICAgICAgIGlmIG1vZGU9PSd6ZXJvJzpleGFjdChhY3R1YWwsZXhwZWN0ZWQpCiAgICAgICAgZWxzZTplbmdpbmVlcmluZ19jbG9zZShhY3R1YWwsZXhwZWN0ZWQpCiAgICAgICAgY2xhc3Nlcy5hcHBlbmQobW9kZSkKICAgICMgTmF0aXZlIGVwaWxvZ3VlIG5lZ2F0aXZlIHdpdG5lc3M6IGFkZGluZyBiaWFzIEFGVEVSIGhhbGYgcm91bmRpbmcgaXMgd3JvbmcuCiAgICB3PXRvcmNoLm9uZXMoMjU2LDE1MzYsZGV2aWNlPSdjdWRhJyxkdHlwZT10b3JjaC5mbG9hdDE2KQogICAgcCxzPUYucXVhbnRpemVfNGJpdCh3LGJsb2Nrc2l6ZT02NCxjb21wcmVzc19zdGF0aXN0aWNzPVRydWUscXVhbnRfdHlwZT0nbmY0JykKICAgIGE9dG9yY2guemVyb3MoNjUsMTUzNixkZXZpY2U9J2N1ZGEnLGR0eXBlPXRvcmNoLmZsb2F0MTYpO2FbOiwwXT0xLjthWzosMV09LjAwMDQKICAgIGI9dG9yY2guZnVsbCgoMjU2LCksLjAwMDIsZGV2aWNlPSdjdWRhJyxkdHlwZT10b3JjaC5mbG9hdDE2KTtkYXRhPShhLHAscyxiKQogICAgZXhwZWN0ZWQ9Y2FsbChkYXRhKQogICAgd2l0aCBnLnRlbXBvcmFyeV9uYXRpdmVfZmFsbGJhY2soRVhQRUNURUQpOmFjdHVhbD1jYWxsKGRhdGEpCiAgICBleGFjdChhY3R1YWwsZXhwZWN0ZWQpCiAgICB3cm9uZz0odG9yY2gubm4uZnVuY3Rpb25hbC5saW5lYXIoYSxGLmRlcXVhbnRpemVfNGJpdChwLHMpKStiKQogICAgaWYgdG9yY2guZXF1YWwod3JvbmcsZXhwZWN0ZWQpOnJhaXNlIFJ1bnRpbWVFcnJvcignQmlhcyBuZWdhdGl2ZSB3aXRuZXNzIGJlY2FtZSB2YWN1b3VzJykKICAgIGlmIE1hdE11bDRCaXQuYmFja3dhcmQgaXMgbm90IGJhY2t3YXJkOnJhaXNlIFJ1bnRpbWVFcnJvcignTmF0aXZlIGJhY2t3YXJkIGlkZW50aXR5IGNoYW5nZWQnKQogICAgdG9yY2guY3VkYS5zeW5jaHJvbml6ZSgpCiAgICByZXR1cm4geydyb3dzJzpyb3dzLCdjb2RlZ2VuJzpjb2RlZ2VuLCdncmFkaWVudHMnOmdyYWRpZW50cywnY2xhc3NfZml4dHVyZXMnOmNsYXNzZXMsCiAgICAgICAgJ2JpYXNfZnAzMl9iZWZvcmVfaGFsZl9uZWdhdGl2ZV93aXRuZXNzJzpUcnVlLCdjb3VudHMnOmRpY3QoZy5DT1VOVFMpLAogICAgICAgICduYXRpdmVfYmFja3dhcmRfaWRlbnRpdHlfcHJlc2VydmVkJzpUcnVlLCd0YXNrX3F1YWxpdHknOidVTktOT1dOJ30KCmRlZiBtYWluKCk6CiAgICBwYXJzZXI9YXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKTtwYXJzZXIuYWRkX2FyZ3VtZW50KCctLW91dCcscmVxdWlyZWQ9VHJ1ZSk7YXJncz1wYXJzZXIucGFyc2VfYXJncygpCiAgICByZXBvcnQ9eydzdGF0dXMnOidGQUlMRURfUFJFRkxJR0hUX09OTFknLCdzY29wZSc6J01JQ1JPQkVOQ0hfT05MWScsJ3RyYWluaW5nX2V4ZWN1dGVkJzpGYWxzZSwnc2NvcmUnOk5vbmUsJ2FkbWlzc2lvbic6J1VOS05PV04nfQogICAgdHJ5OnJlcG9ydFsnY2hlY2tzJ109cnVuKCk7cmVwb3J0WydzdGF0dXMnXT0nUEFTU0VEX1BSRUZMSUdIVF9PTkxZJwogICAgZXhjZXB0IEJhc2VFeGNlcHRpb24gYXMgZTpyZXBvcnQudXBkYXRlKGVycm9yPXJlcHIoZSksdHJhY2ViYWNrPXRyYWNlYmFjay5mb3JtYXRfZXhjKCkscHJvZ3Jlc3M9ZGljdChQUk9HUkVTUykpCiAgICBQYXRoKGFyZ3Mub3V0KS53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVwb3J0LGluZGVudD0yLGFsbG93X25hbj1GYWxzZSkrJ1xuJykKICAgIHByaW50KGpzb24uZHVtcHMoeydzdGF0dXMnOnJlcG9ydFsnc3RhdHVzJ10sJ291dHB1dCc6YXJncy5vdXR9KSxmbHVzaD1UcnVlKQogICAgcmFpc2UgU3lzdGVtRXhpdChyZXBvcnRbJ3N0YXR1cyddIT0nUEFTU0VEX1BSRUZMSUdIVF9PTkxZJykKaWYgX19uYW1lX189PSdfX21haW5fXyc6bWFpbigpCg==', 'process_guard.py': 'IiIiSXNvbGF0ZWQgTGludXggc3VicmVhcGVyIGd1YXJkaWFuLiBPd25zIGNoaWxkcmVuIGFuZCBob2xkcyB0aGUgR1BVIGxvY2suCgpUaGUgZ3VhcmRpYW4gd2F0Y2hlcyBhIHBhcmVudC1saXZlbmVzcyBwaXBlOiBkcml2ZXIgU0lHS0lMTCBvciBvdGhlciBhYnJ1cHQKZGVhdGggY2xvc2VzIHRoZSBwaXBlIGFuZCB0cmlnZ2VycyBxdWFyYW50aW5lZCBjbGVhbnVwLiBQSURGRCBzaWduYWxpbmcgaW4gdGhlCnNoYXJlZCB0YXNrIGhlbHBlciBwcmV2ZW50cyBQSUQgcmV1c2UgcmFjZXM7IGRlc2NlbmRhbnRzIGFyZSBvbmx5IHRoaXMgZGVkaWNhdGVkCmd1YXJkaWFuJ3MgY2hpbGRyZW4sIGluY2x1ZGluZyBuZXdseSBhZG9wdGVkIG9ycGhhbnMgZnJvbSBuZXN0ZWQgc2Vzc2lvbnMuClRoaXMgaXMgdGFzayBvcmNoZXN0cmF0aW9uLCBuZXZlciBzb2x1dGlvbi8gd29yayBvciBrZXJuZWwgcGFydGljaXBhdGlvbi4KIiIiCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHNlbGVjdAppbXBvcnQgc2lnbmFsCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKCmltcG9ydCBvd25lZF9wcm9jZXNzX3RyZWUgYXMgdHJlZQoKCmRlZiBxdWFyYW50aW5lKHBhdGgsIHJlYXNvbik6CiAgICAiIiJGaXJzdCBmYWlsdXJlIHdpbnM7IG5ldmVyIG92ZXJ3cml0ZSBvciBhdXRvbWF0aWNhbGx5IGNsZWFyIGEgbWFya2VyLiIiIgogICAgdHJ5OgogICAgICAgIGRlc2NyaXB0b3IgPSBvcy5vcGVuKHBhdGgsIG9zLk9fV1JPTkxZIHwgb3MuT19DUkVBVCB8IG9zLk9fRVhDTCB8IG9zLk9fTk9GT0xMT1csIDBvNjAwKQogICAgZXhjZXB0IEZpbGVFeGlzdHNFcnJvcjoKICAgICAgICByZXR1cm4KICAgIHdpdGggb3MuZmRvcGVuKGRlc2NyaXB0b3IsICJ3IikgYXMgc3RyZWFtOgogICAgICAgIGpzb24uZHVtcCh7InN0YXR1cyI6ICJPUEVSQVRPUl9SRUNPTkNJTElBVElPTl9SRVFVSVJFRCIsICJyZWFzb24iOiByZWFzb24sCiAgICAgICAgICAgICAgICAgICAiZ3VhcmRpYW5fcGlkIjogb3MuZ2V0cGlkKCksICJjcmVhdGVkX3VuaXgiOiB0aW1lLnRpbWUoKX0sIHN0cmVhbSkKICAgICAgICBzdHJlYW0ud3JpdGUoIlxuIikKICAgICAgICBzdHJlYW0uZmx1c2goKQogICAgICAgIG9zLmZzeW5jKHN0cmVhbS5maWxlbm8oKSkKCgpkZWYgX3dyaXRlX3JlY2VpcHQocGF0aCwgdmFsdWUpOgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIHBlbmRpbmcgPSBwYXRoLndpdGhfbmFtZShwYXRoLm5hbWUgKyAiLnBlbmRpbmciKQogICAgcGVuZGluZy53cml0ZV90ZXh0KGpzb24uZHVtcHModmFsdWUsIGluZGVudD0yKSArICJcbiIpCiAgICBwZW5kaW5nLnJlcGxhY2UocGF0aCkKCgpkZWYgZ3VhcmRpYW5fbWFpbigpOgogICAgY29uZmlnID0ganNvbi5sb2FkcyhzeXMuc3RkaW4ucmVhZGxpbmUoKSkKICAgIHJlYWRfZmQsIGxvY2tfZmQgPSBjb25maWdbInBhcmVudF9waXBlX2ZkIl0sIGNvbmZpZ1sibG9ja19mZCJdCiAgICBzdGFydGVkLCByZXF1ZXN0ZWRfc2lnbmFsLCBwcm9jZXNzID0gdGltZS5tb25vdG9uaWMoKSwgW05vbmVdLCBOb25lCiAgICByZXN1bHQgPSB7ImV4aXRfY29kZSI6IDEyNSwgInJvb3RfZXhpdF9jb2RlIjogTm9uZSwgInRpbWVkX291dCI6IEZhbHNlLAogICAgICAgICAgICAgICJpbnRlcnJ1cHRlZCI6IEZhbHNlLCAiY2xlYW51cF9jb25maXJtZWQiOiBGYWxzZSwKICAgICAgICAgICAgICAicmVhc29uIjogTm9uZSwgImd1YXJkaWFuX3BpZCI6IG9zLmdldHBpZCgpfQogICAgZGVmIHJlcXVlc3Qoc2lnbnVtLCBmcmFtZSk6CiAgICAgICAgcmVxdWVzdGVkX3NpZ25hbFswXSA9IHNpZ251bQogICAgZm9yIHNpZyBpbiAoc2lnbmFsLlNJR0lOVCwgc2lnbmFsLlNJR1RFUk0pOgogICAgICAgIHNpZ25hbC5zaWduYWwoc2lnLCByZXF1ZXN0KQogICAgdHJ5OgogICAgICAgIHRyZWUucHJlcGFyZSgpICAjIHJlcXVpcmVkIGJlZm9yZSBhbnkgY2hpbGQ7IG5ldmVyIHNpbGVudGx5IGRlZ3JhZGUKICAgICAgICB3aXRoIFBhdGgoY29uZmlnWyJsb2ciXSkub3BlbigieCIpIGFzIHN0cmVhbToKICAgICAgICAgICAgcHJvY2VzcyA9IHN1YnByb2Nlc3MuUG9wZW4oY29uZmlnWyJjb21tYW5kIl0sIGN3ZD1jb25maWdbImN3ZCJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzdGRvdXQ9c3RyZWFtLCBzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0YXJ0X25ld19zZXNzaW9uPVRydWUpCiAgICAgICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgICAgICByZWFkeSwgXywgXyA9IHNlbGVjdC5zZWxlY3QoW3JlYWRfZmRdLCBbXSwgW10sIDAuMDUpCiAgICAgICAgICAgICAgICBpZiByZWFkeSBhbmQgb3MucmVhZChyZWFkX2ZkLCAxKSA9PSBiIiI6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249InBhcmVudF9saXZlbmVzc19waXBlX2Nsb3NlZCIsIGludGVycnVwdGVkPVRydWUpCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgIGlmIHJlcXVlc3RlZF9zaWduYWxbMF0gaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249Imd1YXJkaWFuX3NpZ25hbF8iICsgc3RyKHJlcXVlc3RlZF9zaWduYWxbMF0pLCBpbnRlcnJ1cHRlZD1UcnVlKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICBpZiB0aW1lLm1vbm90b25pYygpIC0gc3RhcnRlZCA+PSBjb25maWdbInRpbWVvdXQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHQudXBkYXRlKHJlYXNvbj0ib3V0ZXJfdGltZW91dCIsIHRpbWVkX291dD1UcnVlKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICByZXR1cm5jb2RlID0gcHJvY2Vzcy5wb2xsKCkKICAgICAgICAgICAgICAgIGlmIHJldHVybmNvZGUgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0WyJyb290X2V4aXRfY29kZSJdID0gcmV0dXJuY29kZQogICAgICAgICAgICAgICAgICAgIGlmIHRyZWUuZGVzY2VuZGFudHMoKToKICAgICAgICAgICAgICAgICAgICAgICAgcmVzdWx0WyJyZWFzb24iXSA9ICJyb290X2V4aXRlZF93aXRoX293bmVkX2Rlc2NlbmRhbnRzIgogICAgICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgICAgIHJlc3VsdC51cGRhdGUoZXhpdF9jb2RlPXJldHVybmNvZGUsIGNsZWFudXBfY29uZmlybWVkPVRydWUpCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgaWYgcmVzdWx0WyJyZWFzb24iXSBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIHF1YXJhbnRpbmUoY29uZmlnWyJtYXJrZXIiXSwgcmVzdWx0WyJyZWFzb24iXSkKICAgICAgICAgICAgICAgICMgY2xlYW51cCBpZ25vcmVzIHJlcGVhdGVkIElOVC9URVJNLCBzdG9wcy9yZXNjYW5zL2tpbGxzIGJ5CiAgICAgICAgICAgICAgICAjIHBpZGZkIGFuZCB3YWl0cy9yZWFwcyB3aXRob3V0IGFueSB0aW1lb3V0LXVubG9jayBlc2NhcGUuCiAgICAgICAgICAgICAgICB0cmVlLmNsZWFudXAocHJvY2VzcykKICAgICAgICAgICAgICAgIHJlc3VsdC51cGRhdGUocm9vdF9leGl0X2NvZGU9cHJvY2Vzcy5yZXR1cm5jb2RlLCBjbGVhbnVwX2NvbmZpcm1lZD1UcnVlKQogICAgICAgICAgICAgICAgaWYgcmVzdWx0WyJ0aW1lZF9vdXQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHRbImV4aXRfY29kZSJdID0gMTI0CiAgICAgICAgICAgICAgICBlbGlmIHJlc3VsdFsiaW50ZXJydXB0ZWQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHRbImV4aXRfY29kZSJdID0gMTMwCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249Imd1YXJkaWFuX2V4Y2VwdGlvbjoiICsgcmVwcihleGMpLCBleGl0X2NvZGU9MTI1KQogICAgICAgIHRyeToKICAgICAgICAgICAgcXVhcmFudGluZShjb25maWdbIm1hcmtlciJdLCByZXN1bHRbInJlYXNvbiJdKQogICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgIGlmIHByb2Nlc3MgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICB0cmVlLmNsZWFudXAocHJvY2VzcykKICAgICAgICAgICAgcmVzdWx0WyJjbGVhbnVwX2NvbmZpcm1lZCJdID0gbm90IHRyZWUuZGVzY2VuZGFudHMoKQogICAgZmluYWxseToKICAgICAgICByZXN1bHRbIm91dGVyX3dhbGxfcyJdID0gdGltZS5tb25vdG9uaWMoKSAtIHN0YXJ0ZWQKICAgICAgICBfd3JpdGVfcmVjZWlwdChjb25maWdbInJlY2VpcHQiXSwgcmVzdWx0KQogICAgICAgIG9zLmNsb3NlKHJlYWRfZmQpCiAgICAgICAgIyBUaGUgaW5oZXJpdGVkIG9wZW4tZmlsZSBkZXNjcmlwdGlvbiByZXRhaW5zIHRoZSBjYW5vbmljYWwgZmxvY2sgZXZlbgogICAgICAgICMgaWYgdGhlIGRyaXZlciBoYXMgZGllZC4gQ2xvc2UgT05MWSBhZnRlciBvd25lZC10cmVlIGNsZWFudXAgY29tcGxldGVzLgogICAgICAgIG9zLmNsb3NlKGxvY2tfZmQpCiAgICByZXR1cm4gcmVzdWx0WyJleGl0X2NvZGUiXQoKCmRlZiBydW4oY29tbWFuZCwgKiwgY3dkLCBlbnYsIGxvZywgdGltZW91dCwgbG9ja19mZCwKICAgICAgICBtYXJrZXI9Ii90bXAvc291cC1ncHUwLmxvY2suYmxvY2tlZCIpOgogICAgIiIiRGVkaWNhdGVkIGRyaXZlciBBUEkuIEFueSBCYXNlRXhjZXB0aW9uIGFza3MgZ3VhcmRpYW4gY2xlYW51cCBhbmQgd2FpdHMuIiIiCiAgICByZWNlaXB0ID0gUGF0aChzdHIobG9nKSArICIucHJvY2Vzcy5qc29uIikucmVzb2x2ZSgpCiAgICBpZiByZWNlaXB0LmV4aXN0cygpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiUHJvY2VzcyByZWNlaXB0IGFscmVhZHkgZXhpc3RzOyByZWZ1c2UgdG8gb3ZlcndyaXRlIikKICAgIHJlYWRfZmQsIHdyaXRlX2ZkID0gb3MucGlwZSgpCiAgICBjb25maWcgPSB7ImNvbW1hbmQiOiBbc3RyKHgpIGZvciB4IGluIGNvbW1hbmRdLCAiY3dkIjogc3RyKFBhdGgoY3dkKS5yZXNvbHZlKCkpLAogICAgICAgICAgICAgICJsb2ciOiBzdHIoUGF0aChsb2cpLnJlc29sdmUoKSksICJ0aW1lb3V0IjogdGltZW91dCwKICAgICAgICAgICAgICAicmVjZWlwdCI6IHN0cihyZWNlaXB0KSwgInBhcmVudF9waXBlX2ZkIjogcmVhZF9mZCwKICAgICAgICAgICAgICAibG9ja19mZCI6IGxvY2tfZmQsICJtYXJrZXIiOiBzdHIobWFya2VyKX0KICAgIGd1YXJkaWFuID0gTm9uZQogICAgaW50ZXJydXB0ZWQgPSBOb25lCiAgICBkZWYgcmVxdWVzdChzaWdudW0sIGZyYW1lKToKICAgICAgICByYWlzZSBJbnRlcnJ1cHRlZEVycm9yKCJEcml2ZXIgcmVjZWl2ZWQgc2lnbmFsICIgKyBzdHIoc2lnbnVtKSkKICAgIG9sZF9oYW5kbGVycyA9IHtzaWc6IHNpZ25hbC5zaWduYWwoc2lnLCByZXF1ZXN0KSBmb3Igc2lnIGluIChzaWduYWwuU0lHSU5ULCBzaWduYWwuU0lHVEVSTSl9CiAgICB0cnk6CiAgICAgICAgZ3VhcmRpYW4gPSBzdWJwcm9jZXNzLlBvcGVuKFtzeXMuZXhlY3V0YWJsZSwgc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKSksICItLWd1YXJkaWFuIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0ZGluPXN1YnByb2Nlc3MuUElQRSwgZW52PWVudiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcGFzc19mZHM9KHJlYWRfZmQsIGxvY2tfZmQpLCBzdGFydF9uZXdfc2Vzc2lvbj1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0ZXh0PVRydWUpCiAgICAgICAgb3MuY2xvc2UocmVhZF9mZCkKICAgICAgICByZWFkX2ZkID0gTm9uZQogICAgICAgIGd1YXJkaWFuLnN0ZGluLndyaXRlKGpzb24uZHVtcHMoY29uZmlnKSArICJcbiIpCiAgICAgICAgZ3VhcmRpYW4uc3RkaW4uY2xvc2UoKQogICAgICAgIGd1YXJkaWFuLndhaXQoKQogICAgZXhjZXB0IEJhc2VFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIGludGVycnVwdGVkID0gcmVwcihleGMpCiAgICAgICAgIyBJZ25vcmUgcmVwZWF0ZWQgc3RvcCBzaWduYWxzIHVudGlsIHRoZSBndWFyZGlhbiBjb25maXJtcyBjbGVhbnVwLgogICAgICAgIGZvciBzaWcgaW4gb2xkX2hhbmRsZXJzOgogICAgICAgICAgICBzaWduYWwuc2lnbmFsKHNpZywgc2lnbmFsLlNJR19JR04pCiAgICAgICAgaWYgd3JpdGVfZmQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIG9zLmNsb3NlKHdyaXRlX2ZkKQogICAgICAgICAgICB3cml0ZV9mZCA9IE5vbmUKICAgICAgICBpZiBndWFyZGlhbiBpcyBub3QgTm9uZToKICAgICAgICAgICAgaWYgZ3VhcmRpYW4uc3RkaW4gaXMgbm90IE5vbmUgYW5kIG5vdCBndWFyZGlhbi5zdGRpbi5jbG9zZWQ6CiAgICAgICAgICAgICAgICBndWFyZGlhbi5zdGRpbi5jbG9zZSgpCiAgICAgICAgICAgIGd1YXJkaWFuLndhaXQoKQogICAgZmluYWxseToKICAgICAgICBpZiByZWFkX2ZkIGlzIG5vdCBOb25lOgogICAgICAgICAgICBvcy5jbG9zZShyZWFkX2ZkKQogICAgICAgIGlmIHdyaXRlX2ZkIGlzIG5vdCBOb25lOgogICAgICAgICAgICBvcy5jbG9zZSh3cml0ZV9mZCkKICAgICAgICBmb3Igc2lnLCBoYW5kbGVyIGluIG9sZF9oYW5kbGVycy5pdGVtcygpOgogICAgICAgICAgICBzaWduYWwuc2lnbmFsKHNpZywgaGFuZGxlcikKICAgIGlmIG5vdCByZWNlaXB0LmlzX2ZpbGUoKToKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImd1YXJkaWFuX21pc3NpbmdfcmVjZWlwdCIpCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJHdWFyZGlhbiByZWNlaXB0IG1pc3Npbmc7IEdQVSBxdWFyYW50aW5lIHJlcXVpcmVzIG9wZXJhdG9yIHJlY29uY2lsaWF0aW9uIikKICAgIHJlc3VsdCA9IGpzb24ubG9hZHMocmVjZWlwdC5yZWFkX3RleHQoKSkKICAgIGlmIGludGVycnVwdGVkIGlzIG5vdCBOb25lOgogICAgICAgIHJlc3VsdC51cGRhdGUocGFyZW50X2V4Y2VwdGlvbj1pbnRlcnJ1cHRlZCwgaW50ZXJydXB0ZWQ9VHJ1ZSkKICAgICAgICBpZiByZXN1bHRbImV4aXRfY29kZSJdID09IDA6CiAgICAgICAgICAgIHJlc3VsdFsiZXhpdF9jb2RlIl0gPSAxMzAKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImRyaXZlcl9pbnRlcnJ1cHRpb246IiArIGludGVycnVwdGVkKQogICAgaWYgbm90IHJlc3VsdFsiY2xlYW51cF9jb25maXJtZWQiXToKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImd1YXJkaWFuX3VuY2VydGFpbl9jbGVhbnVwIikKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkd1YXJkaWFuIGNsZWFudXAgdW5jZXJ0YWluOyBxdWFyYW50aW5lIHJldGFpbmVkIikKICAgIHJldHVybiByZXN1bHQKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgaWYgc3lzLmFyZ3ZbMTpdICE9IFsiLS1ndWFyZGlhbiJdOgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoIkludGVybmFsIGd1YXJkaWFuIGVudHJ5cG9pbnQgb25seSIpCiAgICByYWlzZSBTeXN0ZW1FeGl0KGd1YXJkaWFuX21haW4oKSkK', 'owned_process_tree.py': 'IyBTaGFyZWQgdGFzayBvcmNoZXN0cmF0aW9uIGZyb20gdGhlIExvUkEgbGFuZTsgbm8ga2VybmVsIG93bmVyc2hpcCBjcmVkaXQuCiIiIkxpbnV4LW9ubHkgZGVkaWNhdGVkLXJ1bm5lciBjbGVhbnVwOyBubyBHUFUgaW1wb3J0cyBvciBiYWNrZ3JvdW5kIGFjdGl2aXR5LgoKT3JpZ2luYWwgdGFzay1vcmNoZXN0cmF0aW9uIGhlbHBlci4gTm90IHBhcnQgb2Ygc29sdXRpb24vIG9yIG93bi1rZXJuZWwgY3JlZGl0LgoiIiIKaW1wb3J0IGN0eXBlcwppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzaWduYWwKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQoKCmRlZiBwcmVwYXJlKCk6CiAgICAiIiJGYWlsIGJlZm9yZSBHUFUgd29yayBpZiByYWNlLXNhZmUgcHJvY2VzcyBvd25lcnNoaXAgaXMgdW5hdmFpbGFibGUuIiIiCiAgICBpZiBub3QgaGFzYXR0cihvcywncGlkZmRfb3BlbicpIG9yIG5vdCBoYXNhdHRyKHNpZ25hbCwncGlkZmRfc2VuZF9zaWduYWwnKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0xpbnV4IHBpZGZkIHByb2Nlc3Mgb3duZXJzaGlwIHJlcXVpcmVkJykKICAgIGZkPW9zLnBpZGZkX29wZW4ob3MuZ2V0cGlkKCkpO29zLmNsb3NlKGZkKQogICAgbGliYz1jdHlwZXMuQ0RMTChOb25lLHVzZV9lcnJubz1UcnVlKQogICAgaWYgbGliYy5wcmN0bCgzNiwxLDAsMCwwKSE9MDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0Nhbm5vdCBlbmFibGUgY2hpbGQgc3VicmVhcGVyOyBubyBHUFUgam9iIHN0YXJ0ZWQnKQogICAgZW5hYmxlZD1jdHlwZXMuY19pbnQoMCkKICAgIGlmIGxpYmMucHJjdGwoMzcsY3R5cGVzLmJ5cmVmKGVuYWJsZWQpLDAsMCwwKSE9MCBvciBlbmFibGVkLnZhbHVlIT0xOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignQ2hpbGQgc3VicmVhcGVyIHZlcmlmaWNhdGlvbiBmYWlsZWQnKQoKCmRlZiBpZGVudGl0eShwaWQpOgogICAgdHJ5OgogICAgICAgIGZpZWxkcz1QYXRoKGYnL3Byb2Mve3BpZH0vc3RhdCcpLnJlYWRfdGV4dCgpLnJzcGxpdCgnKScsMSlbMV0uc3BsaXQoKQogICAgICAgIHJldHVybiBpbnQoZmllbGRzWzFdKSxpbnQoZmllbGRzWzE5XSksZmllbGRzWzBdCiAgICBleGNlcHQgKE9TRXJyb3IsVmFsdWVFcnJvcixJbmRleEVycm9yKTpyZXR1cm4gTm9uZQoKCmRlZiBkZXNjZW5kYW50cygpOgogICAgcmVjb3Jkcz17fQogICAgZm9yIHAgaW4gUGF0aCgnL3Byb2MnKS5pdGVyZGlyKCk6CiAgICAgICAgaWYgcC5uYW1lLmlzZGlnaXQoKToKICAgICAgICAgICAgcmVjPWlkZW50aXR5KGludChwLm5hbWUpKQogICAgICAgICAgICBpZiByZWMgaXMgbm90IE5vbmU6cmVjb3Jkc1tpbnQocC5uYW1lKV09cmVjCiAgICBvd25lZD17b3MuZ2V0cGlkKCl9CiAgICB3aGlsZSBUcnVlOgogICAgICAgIGFkZD17cGlkIGZvciBwaWQsKHBhcmVudCxfLF8pIGluIHJlY29yZHMuaXRlbXMoKSBpZiBwYXJlbnQgaW4gb3duZWR9LW93bmVkCiAgICAgICAgaWYgbm90IGFkZDpicmVhawogICAgICAgIG93bmVkLnVwZGF0ZShhZGQpCiAgICByZXR1cm4ge3BpZDpyZWNvcmRzW3BpZF0gZm9yIHBpZCBpbiBvd25lZCBpZiBwaWQhPW9zLmdldHBpZCgpIGFuZCBwaWQgaW4gcmVjb3Jkc30KCgpkZWYgc2lnbmFsX2V4YWN0KHBpZCxiaXJ0aCxzaWcpOgogICAgIiIiTmV2ZXIgc2lnbmFsIGEgcmVjeWNsZWQgUElELiBUaGUgRkQgcGlucyBpZGVudGl0eSB0aHJvdWdoIHRoZSBzZW5kLiIiIgogICAgdHJ5OmZkPW9zLnBpZGZkX29wZW4ocGlkKQogICAgZXhjZXB0IFByb2Nlc3NMb29rdXBFcnJvcjpyZXR1cm4KICAgIHRyeToKICAgICAgICBub3c9aWRlbnRpdHkocGlkKQogICAgICAgIGlmIG5vdyBpcyBub3QgTm9uZSBhbmQgbm93WzFdPT1iaXJ0aDoKICAgICAgICAgICAgdHJ5OnNpZ25hbC5waWRmZF9zZW5kX3NpZ25hbChmZCxzaWcsTm9uZSwwKQogICAgICAgICAgICBleGNlcHQgUHJvY2Vzc0xvb2t1cEVycm9yOnBhc3MKICAgIGZpbmFsbHk6b3MuY2xvc2UoZmQpCgoKZGVmIF90cnlfc2lnbmFsKHBpZCxiaXJ0aCxzaWcpOgogICAgdHJ5OnNpZ25hbF9leGFjdChwaWQsYmlydGgsc2lnKQogICAgZXhjZXB0IE9TRXJyb3IgYXMgZXhjOgogICAgICAgICMgRG8gbm90IHVud2luZC9yZWxlYXNlIEdQVSBvd25lcnNoaXAgd2hlbiBjbGVhbnVwIGlzIHVuY2VydGFpbi4KICAgICAgICBxdWFyYW50aW5lKCdDbGVhbnVwIHNpZ25hbCBmYWlsZWQ6ICcrcmVwcihleGMpKQogICAgICAgIHByaW50KCdTT1VQX0NMRUFOVVBfUEVORElORyAnK3JlcHIoZXhjKSxmaWxlPXN5cy5zdGRlcnIsZmx1c2g9VHJ1ZSkKCgpkZWYgY2xlYW51cChwcm9jKToKICAgIHNhdmVkPXtzOnNpZ25hbC5zaWduYWwocyxzaWduYWwuU0lHX0lHTikgZm9yIHMgaW4gKHNpZ25hbC5TSUdJTlQsc2lnbmFsLlNJR1RFUk0pfQogICAgdHJ5OgogICAgICAgIGZyb3plbj1zZXQoKQogICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgIG9ic2VydmVkPWRlc2NlbmRhbnRzKCkKICAgICAgICAgICAgbmV3PXsocGlkLHZbMV0pIGZvciBwaWQsdiBpbiBvYnNlcnZlZC5pdGVtcygpfS1mcm96ZW4KICAgICAgICAgICAgaWYgbm90IG5ldzpicmVhawogICAgICAgICAgICBmb3IgcGlkLGJpcnRoIGluIG5ldzpfdHJ5X3NpZ25hbChwaWQsYmlydGgsc2lnbmFsLlNJR1NUT1ApCiAgICAgICAgICAgIGZyb3plbi51cGRhdGUobmV3KQogICAgICAgIGZvciBwaWQsYmlydGggaW4gZnJvemVuOl90cnlfc2lnbmFsKHBpZCxiaXJ0aCxzaWduYWwuU0lHS0lMTCkKICAgICAgICBwcm9jLndhaXQoKQogICAgICAgICMgTm8gdGltZW91dC1iYXNlZCByZWxlYXNlOiBpZiBhIEdQVSBjaGlsZCBpcyBzdHVjayBpbiBrZXJuZWwgZXhpdCwKICAgICAgICAjIHJldGFpbiB0aGlzIHN1cGVydmlzb3IgYW5kIGl0cyBsb2NrIHVudGlsIGV4aXQgaXMgYWN0dWFsbHkgb2JzZXJ2ZWQuCiAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBwaWQsXz1vcy53YWl0cGlkKC0xLG9zLldOT0hBTkcpCiAgICAgICAgICAgICAgICAgICAgaWYgbm90IHBpZDpicmVhawogICAgICAgICAgICAgICAgZXhjZXB0IENoaWxkUHJvY2Vzc0Vycm9yOmJyZWFrCiAgICAgICAgICAgIHJlbWFpbmluZz1kZXNjZW5kYW50cygpCiAgICAgICAgICAgIGlmIG5vdCByZW1haW5pbmc6YnJlYWsKICAgICAgICAgICAgZm9yIHBpZCwoXyxiaXJ0aCxfKSBpbiByZW1haW5pbmcuaXRlbXMoKTpfdHJ5X3NpZ25hbChwaWQsYmlydGgsc2lnbmFsLlNJR0tJTEwpCiAgICAgICAgICAgIHRpbWUuc2xlZXAoLjA1KQogICAgZmluYWxseToKICAgICAgICBmb3Igc2lnLGhhbmRsZXIgaW4gc2F2ZWQuaXRlbXMoKTpzaWduYWwuc2lnbmFsKHNpZyxoYW5kbGVyKQoKCmRlZiBydW4oYXJndiwqLGN3ZCxlbnYsc3Rkb3V0LHRpbWVvdXQ9Tm9uZSxzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQpOgogICAgIiIiUnVuIGluIGRlZGljYXRlZCBzdXBlcnZpc29yIG9ubHk7IGNsZWFudXAgbmVzdGVkIHNlc3Npb25zIG9uIGFueSBmYWlsdXJlLiIiIgogICAgcHJvYz1zdWJwcm9jZXNzLlBvcGVuKGFyZ3YsY3dkPWN3ZCxlbnY9ZW52LHN0ZG91dD1zdGRvdXQsc3RkZXJyPXN0ZGVycixzdGFydF9uZXdfc2Vzc2lvbj1UcnVlKQogICAgdHJ5OgogICAgICAgIGNvZGU9cHJvYy53YWl0KHRpbWVvdXQ9dGltZW91dCkKICAgICAgICBpZiBkZXNjZW5kYW50cygpOgogICAgICAgICAgICBxdWFyYW50aW5lKCdDaGlsZCBleGl0ZWQgbGVhdmluZyBkZXNjZW5kYW50cycpCiAgICAgICAgICAgIGNsZWFudXAocHJvYykKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdDaGlsZCBleGl0ZWQgbGVhdmluZyBkZXNjZW5kYW50czsgY2xlYW5lZCBhbmQgcmVqZWN0ZWQnKQogICAgICAgIHJldHVybiBjb2RlCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcXVhcmFudGluZSgnT3V0ZXIgdGltZW91dC9pbnRlcnJ1cHRpb24vZXJyb3I6ICcrcmVwcihleGMpKQogICAgICAgIGNsZWFudXAocHJvYykKICAgICAgICByYWlzZQoKCmRlZiBpbnN0YWxsX3NpZ25hbF9oYW5kbGVycygpOgogICAgZGVmIGludGVycnVwdGVkKHNpZ251bSxmcmFtZSk6CiAgICAgICAgcXVhcmFudGluZSgnUnVubmVyIGludGVycnVwdGVkIGJ5IHNpZ25hbCAnK3N0cihzaWdudW0pKQogICAgICAgIHJhaXNlIEludGVycnVwdGVkRXJyb3IoJ1J1bm5lciBpbnRlcnJ1cHRlZCBieSBzaWduYWwgJytzdHIoc2lnbnVtKSkKICAgIGZvciBzaWcgaW4gKHNpZ25hbC5TSUdJTlQsc2lnbmFsLlNJR1RFUk0pOnNpZ25hbC5zaWduYWwoc2lnLGludGVycnVwdGVkKQoKIyBBIGNyYXNoLXN1cnZpdmluZyBtYXJrZXIgaXMgcGxhY2VkIGJlZm9yZSBhbnkgR1BVIHN1YnByb2Nlc3MuIE9ubHkgaXRzIGV4YWN0CiMgb3duZXIgbWF5IGNsZWFyIGl0IG9uIGEgY2xlYW4sIG5vbi1pbnRlcnJ1cHRlZCBleGl0IHdpdGggbm8gbGl2ZSBkZXNjZW5kYW50cy4KIyBJbnRlcnJ1cHRlZC90aW1lZC1vdXQgd29yayBsZWF2ZXMgaXQgZm9yIGV4cGxpY2l0IG9wZXJhdG9yIHJlY29uY2lsaWF0aW9uLgpfTUFSS0VSID0gTm9uZQpfVE9LRU4gPSBOb25lCl9LRUVQX01BUktFUiA9IEZhbHNlCgoKZGVmIGNoZWNrX3F1YXJhbnRpbmUocGF0aCk6CiAgICBpZiBQYXRoKHBhdGgpLmV4aXN0cygpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignR1BVMCBxdWFyYW50aW5lZDogcmVjb25jaWxlICcrc3RyKHBhdGgpKycgYmVmb3JlIGFub3RoZXIgcnVuJykKCgpkZWYgYmVnaW5fcXVhcmFudGluZShwYXRoKToKICAgIGdsb2JhbCBfTUFSS0VSLF9UT0tFTixfS0VFUF9NQVJLRVIKICAgIGltcG9ydCBqc29uLHV1aWQKICAgIF9NQVJLRVI9UGF0aChwYXRoKTtfVE9LRU49dXVpZC51dWlkNCgpLmhleDtfS0VFUF9NQVJLRVI9RmFsc2UKICAgIGZkPW9zLm9wZW4oc3RyKF9NQVJLRVIpLG9zLk9fV1JPTkxZfG9zLk9fQ1JFQVR8b3MuT19FWENMfG9zLk9fTk9GT0xMT1csMG82MDApCiAgICB3aXRoIG9zLmZkb3BlbihmZCwndycpIGFzIG91dDoKICAgICAgICBqc29uLmR1bXAoeydvd25lcl90b2tlbic6X1RPS0VOLCdvd25lcl9waWQnOm9zLmdldHBpZCgpLCdzdGF0dXMnOidPV05FRF9HUFVfV09SS19JTl9QUk9HUkVTUyd9LG91dCkKICAgICAgICBvdXQuZmx1c2goKTtvcy5mc3luYyhvdXQuZmlsZW5vKCkpCgoKZGVmIHF1YXJhbnRpbmUocmVhc29uKToKICAgIGdsb2JhbCBfS0VFUF9NQVJLRVIKICAgIF9LRUVQX01BUktFUj1UcnVlCiAgICBpZiBfTUFSS0VSIGlzIE5vbmU6cmV0dXJuCiAgICBpbXBvcnQganNvbgogICAgdHJ5OnZhbHVlPWpzb24ubG9hZHMoX01BUktFUi5yZWFkX3RleHQoKSkKICAgIGV4Y2VwdCAoT1NFcnJvcixWYWx1ZUVycm9yKTpyZXR1cm4KICAgIGlmIHZhbHVlLmdldCgnb3duZXJfdG9rZW4nKSE9X1RPS0VOOnJldHVybgogICAgdmFsdWUudXBkYXRlKHN0YXR1cz0nQkxPQ0tFRF9PUEVSQVRPUl9SRUNPTkNJTElBVElPTl9SRVFVSVJFRCcscmVhc29uPXN0cihyZWFzb24pKQogICAgX01BUktFUi53cml0ZV90ZXh0KGpzb24uZHVtcHModmFsdWUpKydcbicpCgoKZGVmIGZpbmlzaF9xdWFyYW50aW5lKCk6CiAgICBpZiBfTUFSS0VSIGlzIE5vbmUgb3IgX0tFRVBfTUFSS0VSOnJldHVybgogICAgaWYgZGVzY2VuZGFudHMoKToKICAgICAgICBxdWFyYW50aW5lKCdMaXZlIG93bmVkIGRlc2NlbmRhbnRzIHJlbWFpbiBhdCBzdXBlcnZpc29yIGV4aXQnKQogICAgICAgIHJldHVybgogICAgaW1wb3J0IGpzb24KICAgIHRyeTp2YWx1ZT1qc29uLmxvYWRzKF9NQVJLRVIucmVhZF90ZXh0KCkpCiAgICBleGNlcHQgKE9TRXJyb3IsVmFsdWVFcnJvcik6cmV0dXJuCiAgICBpZiB2YWx1ZS5nZXQoJ293bmVyX3Rva2VuJyk9PV9UT0tFTjpfTUFSS0VSLnVubGluaygpCg==', 'runtime_probe.py': 'IiIiUmVhZC1vbmx5LCBzaG9ydC1saXZlZCBkZXBsb3llZC10ZW1wbGF0ZSBnYXRlLiBOZXZlciBwcmludHMgc2VjcmV0IHZhbHVlcy4iIiIKaW1wb3J0IGltcG9ydGxpYi5tZXRhZGF0YSBhcyBtZXRhZGF0YSxqc29uLG9zLHBsYXRmb3JtLHN1YnByb2Nlc3Msc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApFWFBFQ1RFRD17J3RvcmNoJzonMi4xMS4wK2N1MTI4JywndHJpdG9uJzonMy42LjAnLCdudW1weSc6JzIuMy41Jywnc291cC1jbGknOicwLjc3LjAnLCd0cmFuc2Zvcm1lcnMnOic1LjE2LjEnLCdwZWZ0JzonMC4yMC4wJywndHJsJzonMC4yOS4xJywnYml0c2FuZGJ5dGVzJzonMC41MC4yJywnYWNjZWxlcmF0ZSc6JzEuMTQuMCcsJ2RhdGFzZXRzJzonNC44LjUnfQoKZGVmIG1haW4oKToKIG91dD1QYXRoKHN5cy5hcmd2WzFdKTtyPXsncGFzc2VkJzpGYWxzZSwnZ3B1X3dvcmtsb2FkX2V4ZWN1dGVkJzpGYWxzZSwnc2NvcmUnOk5vbmUsJ2FkbWlzc2lvbic6J1VOS05PV04nfQogdHJ5OgogIHZlcnNpb25zPXtrOm1ldGFkYXRhLnZlcnNpb24oaykgZm9yIGsgaW4gRVhQRUNURUR9CiAgci51cGRhdGUocHl0aG9uPXBsYXRmb3JtLnB5dGhvbl92ZXJzaW9uKCkscGFja2FnZXM9dmVyc2lvbnMsdWlkPW9zLmdldHVpZCgpLGNwdV9hZmZpbml0eT1zb3J0ZWQob3Muc2NoZWRfZ2V0YWZmaW5pdHkoMCkpKQogIGFzc2VydCBwbGF0Zm9ybS5weXRob25fdmVyc2lvbigpPT0nMy4xMy4xNScsJ1B5dGhvbiBwaW4gbWlzbWF0Y2gnCiAgYXNzZXJ0IHZlcnNpb25zPT1FWFBFQ1RFRCwnRXhhY3QgcGFja2FnZSBwaW5zIG1pc21hdGNoJwogIGFzc2VydCBvcy5nZXR1aWQoKT09MTAwMDEsJ0RlcGxveWVkIHJ1bnRpbWUgdWlkIG1pc21hdGNoJwogIGFzc2VydCBsZW4ob3Muc2NoZWRfZ2V0YWZmaW5pdHkoMCkpPT00LCdGb3VyIGFzc2lnbmVkIENQVXMgcmVxdWlyZWQnCiAgZm9yIHAsd2FudGVkIGluIFsoJ21lbW9yeS5tYXgnLHN0cigzMCoxMDI0KiozKSksKCdtZW1vcnkuc3dhcC5tYXgnLCcwJyksKCdjcHVzZXQuY3B1cy5lZmZlY3RpdmUnLCcwLTMnKV06CiAgIGFjdHVhbD0oUGF0aCgnL3N5cy9mcy9jZ3JvdXAnKS9wKS5yZWFkX3RleHQoKS5zdHJpcCgpO3JbcF09YWN0dWFsCiAgIGFzc2VydCBhY3R1YWw9PXdhbnRlZCwnRGVwbG95ZWQgY2dyb3VwIG1pc21hdGNoOiAnK3AKICBhc3NlcnQgbm90IFBhdGgoJy92YXIvcnVuL2RvY2tlci5zb2NrJykuZXhpc3RzKCksJ1VuZXhwZWN0ZWQgRG9ja2VyIHNvY2tldCcKICBhc3NlcnQgbm90IFBhdGgoJy9ldGMvc291cC1xdWV1ZS93b3JrZXIuZW52JykuZXhpc3RzKCksJ1VuZXhwZWN0ZWQgd29ya2VyIGVudmlyb25tZW50IGZpbGUnCiAgYXNzZXJ0IG5vdCBhbnkoJ1RPS0VOJyBpbiBrIGFuZCBrIT0nVE9LRU5JWkVSU19QQVJBTExFTElTTScgZm9yIGsgaW4gb3MuZW52aXJvbiksJ1Rva2VuLWJlYXJpbmcgZW52aXJvbm1lbnQgcmVqZWN0ZWQgd2l0aG91dCByZWFkaW5nIGl0cyB2YWx1ZScKICBhc3NlcnQgbm90IG9zLmVudmlyb24uZ2V0KCdUUklUT05fSU5URVJQUkVUJyksJ0ludGVycHJldGVyIG1vZGUgZm9yYmlkZGVuJwogIGltcG9ydCB0b3JjaCx0cml0b24sbnVtcHkKICBhc3NlcnQgdG9yY2guX192ZXJzaW9uX189PScyLjExLjArY3UxMjgnIGFuZCB0b3JjaC52ZXJzaW9uLmN1ZGE9PScxMi44JwogIGFzc2VydCB0cml0b24uX192ZXJzaW9uX189PSczLjYuMCcgYW5kIG51bXB5Ll9fdmVyc2lvbl9fPT0nMi4zLjUnCiAgYXNzZXJ0IHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCk9PTEgYW5kIHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApPT0nVGVzbGEgVDQnCiAgYXNzZXJ0IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9jYXBhYmlsaXR5KDApPT0oNyw1KQogIHIudXBkYXRlKGdwdV9uYW1lPXRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApLGN1ZGFfYnVpbGQ9dG9yY2gudmVyc2lvbi5jdWRhLHRvcmNoX2dpdD10b3JjaC52ZXJzaW9uLmdpdF92ZXJzaW9uKQogIGZyb20gaHVnZ2luZ2ZhY2VfaHViLmNvbnN0YW50cyBpbXBvcnQgSEZfSFVCX0NBQ0hFCiAgclsncmVzb2x2ZWRfcHVibGljX2h1Yl9jYWNoZSddPXN0cihQYXRoKEhGX0hVQl9DQUNIRSkucmVzb2x2ZSgpKQogIHJbJ3Bhc3NlZCddPVRydWUKIGV4Y2VwdCBCYXNlRXhjZXB0aW9uIGFzIGU6clsnZXJyb3InXT1yZXByKGUpCiBvdXQud3JpdGVfdGV4dChqc29uLmR1bXBzKHIsaW5kZW50PTIpKydcbicpO3ByaW50KGpzb24uZHVtcHMocixpbmRlbnQ9MikpCiByYWlzZSBTeXN0ZW1FeGl0KDAgaWYgclsncGFzc2VkJ10gZWxzZSAxKQppZiBfX25hbWVfXz09J19fbWFpbl9fJzptYWluKCkK', 'official_manifest.json': 'ewogICJoYXJuZXNzL190cmFpbi5weSI6ICI0NWE2MjY0MmRjZWFiMzJjZjNlZmNkYTUwNGRlODdhOWMyNWVhMjhiMTE5YjE4MDZmNTZkMTRmZjExNDZhODlkIiwKICAiaGFybmVzcy9jaGVja19ydWxlcy5weSI6ICI4NzlhYjE4NjE2NzRmNmU3MTRlYWRkMDBhYWYwNDQyMzVkOTg1MjZkZDc1ZmFiMTAzNWRlZWMzNDAxMTIxZWY1IiwKICAiaGFybmVzcy9jb21tb24ucHkiOiAiYzgzMDY2ZjI0ZjI2NmRhYjYyZWJjYzljMmQ0NGY1YTY1OTAyNzYwYjUzZGE5MTJlNjVlYmZlNTExNTc5YzJkYiIsCiAgImhhcm5lc3MvZXZhbF9hZGFwdGVyLnB5IjogIjQ3Y2RkYmI4ZTQ1NzljNjlmOTUzNWIwOTlmZjIzNzFiMWMzYWYwOGJmYmQ5M2QxNGJmNzlhYWRkNzRkMzg0ODIiLAogICJoYXJuZXNzL2V4YW1wbGVfcmVmZXJlbmNlLnB5IjogIjZhMTEwNTVmNGExODIyZTUzOTcxYzIyOGM2ZTNlM2FhNmI4OGVkZDllYjZkYjgxYTM4OGUyNmE3MGEwMzBjZWIiLAogICJoYXJuZXNzL3J1bl90YXNrLnB5IjogImQ5MzVhODk1OWYzNDg5NzczMTNmMmVlZjM2ODcwZmYyNjk5YjliYTAxM2E3OTc4OTE4ZmYzNWUxNDg5NjhhNzkiLAogICJoYXJuZXNzL3Njb3JlLnB5IjogIjczMWM5YjY0NGRjYzQwNjliMGIwNjk2MTZmNDM4MDY0ZDFmN2YzNzZhMjAyZmViMTUyMDFmZDkxMGVjNjg4YzUiLAogICJ0YXNrL3ByZXBhcmVfZGF0YS5weSI6ICJhODU0M2YxMWE1NWFmZjgwZGJkYzcxNGEwZTYyMzkxMjQ0YTUyOTY2MTZlZWEyZTM0ZGMyZWIzZmYzZWMxOGY3IiwKICAidGFzay90YXNrLnlhbWwiOiAiZGQ3MGM3MzlmY2QxNDNlMDZhZWYzYmIxZWZjYzkzODZmNjgyYmYxZGUyNmYzMGUwODQwOWQ4NmE5N2ZmNDdjZSIsCiAgInRlc3RzL2NvbmZ0ZXN0LnB5IjogImE4MmVhYmFiNDY4ZWEwZWYxZTAwM2NjY2Q3YzM4MjdjZTJmNGQwMGQwOWMzMjk0OGEzNGZiNDgyMjFmNzMzN2EiLAogICJ0ZXN0cy90ZXN0X2NhbGxiYWNrLnB5IjogIjE5MTkzMDNhMzY3MGMwYTMzYWJiZmNlOWIzM2VhYjA5MTNiYjY0YTNhMjFjMzMxMmQ4NjUzMWMzMmI1YmY0OTAiLAogICJ0ZXN0cy90ZXN0X2NvbmZpZy5weSI6ICJkYjgwOGM0NDQ3MGQxNDRiN2NmMThiNzMyOTQ5MmY0MWY3NmUzN2NmYzI0MzA2ZTdjYmEwNDg4YmNhOWQ3ODdjIiwKICAidGVzdHMvdGVzdF9mb3JtYXRzLnB5IjogIjVkNjU2ZWQ1MGM1OWY3OTRlOTM0NDczZDQ1OWVhODU2MzVmODkyYmEwMjhlMTkyNDE0NDc0ZTliNWQ2NTkxOWUiLAogICJ0ZXN0cy90ZXN0X2xvYWRlci5weSI6ICIzZGFkMWJlYTJkMzNhYWI5N2U1MmI3Y2Q0NDg4OGQ3NjhiYzk1MTlhZjQ0ZmJlNmNmM2U5NmMyNjAwMjY2MzJmIiwKICAidGVzdHMvdGVzdF90cmFpbmVyX2luaXQucHkiOiAiYjc2MDhkNjJhNDA5ODNiZjdjOWU1YjI3MGUwZTUyN2I2NDFhODM3ZTQ2MmQ2ZGYyOTA5ZmVjODhmNjA3ZTgyMSIKfQo=', 'reviewed_support.py': 'aW1wb3J0IGFyZ3BhcnNlLGZjbnRsLGhhc2hsaWIsanNvbixtYXRoLG9zLHNodXRpbCxzeXMsdGhyZWFkaW5nLHRpbWUsdXVpZCx6aXBmaWxlCgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBwcm9jZXNzX2d1YXJkCgppbXBvcnQgb3duZWRfcHJvY2Vzc190cmVlIGFzIHRyZWUKCmRlZiBzaGEocCk6cmV0dXJuIGhhc2hsaWIuc2hhMjU2KFBhdGgocCkucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKQoKZGVmIHJlYWQocCk6cmV0dXJuIGpzb24ubG9hZHMoUGF0aChwKS5yZWFkX3RleHQoKSkKCmRlZiB3cml0ZShwLHZhbHVlKToKIHA9UGF0aChwKTtwZW5kaW5nPXAud2l0aF9uYW1lKHAubmFtZSsnLnBlbmRpbmcnKTtwZW5kaW5nLndyaXRlX3RleHQoanNvbi5kdW1wcyh2YWx1ZSxpbmRlbnQ9MixhbGxvd19uYW49RmFsc2UpKydcbicpO3BlbmRpbmcucmVwbGFjZShwKQoKZGVmIHZlcmlmeV9vZmZpY2lhbChyZXBvLG1hbmlmZXN0KToKIGlmIG5vdCByZXBvLmlzX2RpcigpIG9yIHJlcG8uaXNfc3ltbGluaygpOnJhaXNlIFJ1bnRpbWVFcnJvcignU09VUF9SRVBPIG11c3QgYmUgYSByZWFsIHByZXBhcmVkIGpvYiBkaXJlY3RvcnknKQogZm9yIG5hbWUsd2FudGVkIGluIG1hbmlmZXN0Lml0ZW1zKCk6CiAgcD1yZXBvL25hbWUKICBpZiBub3QgcC5pc19maWxlKCkgb3IgcC5pc19zeW1saW5rKCkgb3Igc2hhKHApIT13YW50ZWQ6cmFpc2UgUnVudGltZUVycm9yKCdQcm90ZWN0ZWQgb2ZmaWNpYWwgc291cmNlIG1pc21hdGNoOiAnK25hbWUpCiBhY3R1YWw9e3N0cihwLnJlbGF0aXZlX3RvKHJlcG8pKSBmb3IgZm9sZGVyIGluICgnaGFybmVzcycsJ3Rhc2snLCd0ZXN0cycpIGZvciBwIGluIChyZXBvL2ZvbGRlcikucmdsb2IoJyonKQogICAgICAgICBpZiBwLmlzX2ZpbGUoKSBhbmQgJ19fcHljYWNoZV9fJyBub3QgaW4gcC5wYXJ0cyBhbmQgcC5zdWZmaXggbm90IGluICgnLnB5YycsJy5weW8nKX0KIGlmIGFjdHVhbCE9c2V0KG1hbmlmZXN0KTpyYWlzZSBSdW50aW1lRXJyb3IoJ1VuZXhwZWN0ZWQgcHJvdGVjdGVkIG9mZmljaWFsIGZpbGVzJykKCkVYUE9SVF9CVURHRVQ9NTYqMTAyNCoqMgoKRVhQT1JUX09NSVRURUQ9e30KCmRlZiBleHBvcnRfc21hbGwocnVuLGV4cG9ydCk6CiAiIiJSZWFkLW9ubHkgaW5jcmVtZW50YWwgZXhwb3J0OyBubyBhZGFwdGVycywgd2VpZ2h0cyBvciBjb21waWxlciBjYWNoZXMuIiIiCiB1c2VkPXN1bShwLnN0YXQoKS5zdF9zaXplIGZvciBwIGluIGV4cG9ydC5yZ2xvYignKicpIGlmIHAuaXNfZmlsZSgpKQogZXhjbHVkZWQ9eydhZGFwdGVyJywndHJpdG9uX2NhY2hlJywnaW5kdWN0b3JfY2FjaGUnLCdwcmVmbGlnaHRfdHJpdG9uJywncHJlZmxpZ2h0X2luZHVjdG9yJywnb3JpZ2luYWxfc29sdXRpb24nfQogZm9yIGJhc2UsZGlycyxmaWxlcyBpbiBvcy53YWxrKHJ1bik6CiAgZGlyc1s6XT1bZCBmb3IgZCBpbiBkaXJzIGlmIGQgbm90IGluIGV4Y2x1ZGVkIGFuZCBub3QgZC5zdGFydHN3aXRoKCdjaGVja3BvaW50LScpIGFuZCBub3QgKFBhdGgoYmFzZSkvZCkuaXNfc3ltbGluaygpXQogIGZvciBuYW1lIGluIGZpbGVzOgogICBwYXRoPVBhdGgoYmFzZSkvbmFtZTtyZWw9cGF0aC5yZWxhdGl2ZV90byhydW4pCiAgIHRyeToKICAgIGlmIG5vdCBwYXRoLmlzX3N5bWxpbmsoKSBhbmQgcGF0aC5zdWZmaXggaW4gKCcuanNvbicsJy5qc29ubCcsJy5sb2cnLCcueWFtbCcpOgogICAgIGlmIHBhdGguc3RhdCgpLnN0X3NpemU+PTQqMTAyNCoqMjoKICAgICAgRVhQT1JUX09NSVRURURbc3RyKHJlbCldPSdwZXJfZmlsZV9saW1pdF80TWlCJztjb250aW51ZQogICAgIGRlc3Q9ZXhwb3J0LydyYXcnL3JlbDtkZXN0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgICBwcmlvcj1kZXN0LnN0YXQoKS5zdF9zaXplIGlmIGRlc3QuZXhpc3RzKCkgZWxzZSAwCiAgICAgc2l6ZT1wYXRoLnN0YXQoKS5zdF9zaXplCiAgICAgaWYgdXNlZC1wcmlvcitzaXplPkVYUE9SVF9CVURHRVQ6CiAgICAgIEVYUE9SVF9PTUlUVEVEW3N0cihyZWwpXT0nYWdncmVnYXRlX2J1ZGdldF81Nk1pQic7Y29udGludWUKICAgICB0ZW1wPWRlc3Qud2l0aF9uYW1lKGRlc3QubmFtZSsnLmNvcHlpbmcnKTtzaHV0aWwuY29weWZpbGUocGF0aCx0ZW1wKTt0ZW1wLnJlcGxhY2UoZGVzdCk7dXNlZD11c2VkLXByaW9yK3NpemUKICAgZXhjZXB0IChGaWxlTm90Rm91bmRFcnJvcixPU0Vycm9yKTpwYXNzCgpjbGFzcyBFeHBvcnRlcjoKIGRlZiBfX2luaXRfXyhzZWxmLHJ1bixleHBvcnQpOnNlbGYucnVuPXJ1bjtzZWxmLmV4cG9ydD1leHBvcnQ7c2VsZi5zdG9wPXRocmVhZGluZy5FdmVudCgpO3NlbGYudGhyZWFkPU5vbmUKIGRlZiBfX2VudGVyX18oc2VsZik6CiAgZGVmIGxvb3AoKToKICAgd2hpbGUgbm90IHNlbGYuc3RvcC53YWl0KDEpOmV4cG9ydF9zbWFsbChzZWxmLnJ1bixzZWxmLmV4cG9ydCkKICBzZWxmLnRocmVhZD10aHJlYWRpbmcuVGhyZWFkKHRhcmdldD1sb29wLGRhZW1vbj1UcnVlKTtzZWxmLnRocmVhZC5zdGFydCgpO3JldHVybiBzZWxmCiBkZWYgX19leGl0X18oc2VsZiwqZXhjKTpzZWxmLnN0b3Auc2V0KCk7c2VsZi50aHJlYWQuam9pbigpO2V4cG9ydF9zbWFsbChzZWxmLnJ1bixzZWxmLmV4cG9ydCkKCmRlZiBpZGxlX2NoZWNrKGVudixyZXBvLHJ1bixsb2NrZmQpOgogY29kZT0nJydpbXBvcnQganNvbixzdWJwcm9jZXNzLHN5cwpwPXN1YnByb2Nlc3MucnVuKFsnbnZpZGlhLXNtaScsJy0tcXVlcnktY29tcHV0ZS1hcHBzPXBpZCx1c2VkX2dwdV9tZW1vcnknLCctLWZvcm1hdD1jc3Ysbm9oZWFkZXIsbm91bml0cyddLGNhcHR1cmVfb3V0cHV0PVRydWUsdGV4dD1UcnVlLGNoZWNrPVRydWUpCmFzc2VydCBub3QgcC5zdGRvdXQuc3RyaXAoKSwnR1BVIHByb2Nlc3NlcyBzdGlsbCBhY3RpdmUgYWZ0ZXIgcHJvYmUvcHJlZmxpZ2h0JwpyPXN1YnByb2Nlc3MucnVuKFsnbnZpZGlhLXNtaScsJy0tcXVlcnktZ3B1PWluZGV4LG1lbW9yeS51c2VkJywnLS1mb3JtYXQ9Y3N2LG5vaGVhZGVyLG5vdW5pdHMnXSxjYXB0dXJlX291dHB1dD1UcnVlLHRleHQ9VHJ1ZSxjaGVjaz1UcnVlKQpyb3dzPVtsaW5lLnNwbGl0KCcsJykgZm9yIGxpbmUgaW4gci5zdGRvdXQuc3BsaXRsaW5lcygpIGlmIGxpbmUuc3RyaXAoKV0KYXNzZXJ0IHJvd3MgYW5kIHN1bShpbnQoeFsxXSkgZm9yIHggaW4gcm93cyk8MTAwMCwnR1BVIHBoeXNpY2FsIGludmVudG9yeSBub3QgaWRsZScKcHJpbnQoanNvbi5kdW1wcyh7J2lkbGUnOlRydWUsJ3BoeXNpY2FsX21lbW9yeV9yb3dzJzpyb3dzfSkpJycnCiByZXR1cm4gcHJvY2Vzc19ndWFyZC5ydW4oW3N5cy5leGVjdXRhYmxlLCctYycsY29kZV0sY3dkPXJlcG8sZW52PWVudixsb2c9cnVuLydpZGxlLmxvZycsdGltZW91dD0yMCxsb2NrX2ZkPWxvY2tmZCkK', 'worker.py': 'IiIiRnJlc2gtcHJvY2VzcyBxL3Ytc2hhcGVkIGZvcndhcmQgbWljcm9iZW5jaCwgbmV2ZXIgYSB0cmFpbmluZyBiZW5jaG1hcmsuIiIiCmltcG9ydCBhcmdwYXJzZSxjb250ZXh0bGliLGpzb24sb3MsdGltZSx0cmFjZWJhY2sKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpkZWYgbWFpbigpOgogICAgcD1hcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpO3AuYWRkX2FyZ3VtZW50KCctLW1ldGhvZCcsY2hvaWNlcz1bJ25hdGl2ZScsJ2NhbmRpZGF0ZSddLHJlcXVpcmVkPVRydWUpCiAgICBwLmFkZF9hcmd1bWVudCgnLS1vdXQnLHJlcXVpcmVkPVRydWUpO2FyZ3M9cC5wYXJzZV9hcmdzKCk7b3V0PVBhdGgoYXJncy5vdXQpCiAgICBpZiBvdXQuZXhpc3RzKCk6cmFpc2UgRmlsZUV4aXN0c0Vycm9yKG91dCkKICAgIHJlcG9ydD17J3N0YXR1cyc6J0ZBSUxFRF9NSUNST0JFTkNIX09OTFknLCdzY29wZSc6J01JQ1JPQkVOQ0hfT05MWScsJ21ldGhvZCc6YXJncy5tZXRob2QsCiAgICAgICAgJ2dwdV9leGVjdXRlZCc6RmFsc2UsJ3RyYWluaW5nX2V4ZWN1dGVkJzpGYWxzZSwnY29tcGxldGVfZml4ZWRfdGFzayc6RmFsc2UsJ3Jvd3MnOltdLAogICAgICAgICdzY29yZSc6Tm9uZSwnYWRtaXNzaW9uJzonVU5LTk9XTicsJ21ldGhvZF92YXJpYW50cyc6eyduYXRpdmUnOidibmIgbmF0aXZlIGRpc3BhdGNoZXIvZmFsbGJhY2snLAogICAgICAgICAgICAnY2FuZGlkYXRlJzonc2FtZSBuYXRpdmUgZGlzcGF0Y2hlciB3aXRoIGd1YXJkZWQgb3JpZ2luYWwgMTZ4OCBvbmUtd2FycCBmdXNlZCBORjQgTU1BIGZhbGxiYWNrJ30sCiAgICAgICAgJ21lYXN1cmVtZW50X2NvbnRyYWN0Jzp7J2NvbGQnOidzeW5jaHJvbml6ZWQgZmlyc3Qgd3JhcHBlciBjYWxsIHBlciBzaGFwZTsgaW5jbHVkZXMgbmVlZGVkIEpJVCwgYWxsb2NhdGlvbiwgZGlzcGF0Y2ggYW5kIGRldmljZSB3b3JrLCBub3QgcHVyZSBjb21waWxlJywKICAgICAgICAgICAgJ3dhcm0nOicxMiBzZXBhcmF0ZSBDVURBIGV2ZW50IHN0cmVhbSBpbnRlcnZhbHMgYWZ0ZXIgMyB3YXJtdXAgY2FsbHM7IFB5dGhvbiBsYXVuY2ggZ2FwcyBpbmNsdWRlZCcsCiAgICAgICAgICAgICdvcmRlcic6J25hdGl2ZSB0aGVuIGNhbmRpZGF0ZSBpbiBzZXBhcmF0ZSBwcm9jZXNzZXMgYW5kIHByaXZhdGUgY2FjaGVzOyBub3QgY291bnRlcmJhbGFuY2VkJywKICAgICAgICAgICAgJ2lucHV0cyc6J3N5bnRoZXRpYyBzY2FsZWQgcmFuZG9tIHEvdiBzaGFwZXM7IG5vdCBjYXB0dXJlZCB0YXNrIGFjdGl2YXRpb24gZGlzdHJpYnV0aW9ucycsCiAgICAgICAgICAgICdub190YXNrX3NwZWVkX29yX3NoYXJlX2luZmVyZW5jZSc6VHJ1ZX19CiAgICB0cnk6CiAgICAgICAgaWYgb3MuZW52aXJvbi5nZXQoJ1RSSVRPTl9JTlRFUlBSRVQnKTpyYWlzZSBSdW50aW1lRXJyb3IoJ0ludGVycHJldGVyIGZvcmJpZGRlbicpCiAgICAgICAgY2FjaGU9UGF0aChvcy5lbnZpcm9uWydUUklUT05fQ0FDSEVfRElSJ10pCiAgICAgICAgaWYgbm90IGNhY2hlLmlzX2RpcigpIG9yIGFueShjYWNoZS5pdGVyZGlyKCkpOnJhaXNlIFJ1bnRpbWVFcnJvcignUHJpdmF0ZSBlbXB0eSBpbml0aWFsIGNhY2hlIHJlcXVpcmVkJykKICAgICAgICBpbXBvcnQgdG9yY2gKICAgICAgICBmcm9tIHNvbHV0aW9uLmZpeHR1cmVzIGltcG9ydCBFWFBFQ1RFRCxTSEFQRVMsZml4dHVyZSxjYWxsLGVuZ2luZWVyaW5nX2Nsb3NlCiAgICAgICAgZnJvbSBzb2x1dGlvbiBpbXBvcnQgZ3VhcmRlZF9mb3J3YXJkIGFzIGcKICAgICAgICBjdHg9bGFtYmRhOmcudGVtcG9yYXJ5X25hdGl2ZV9mYWxsYmFjayhFWFBFQ1RFRCkgaWYgYXJncy5tZXRob2Q9PSdjYW5kaWRhdGUnIGVsc2UgY29udGV4dGxpYi5udWxsY29udGV4dCgpCiAgICAgICAgZm9yIG0sbiBpbiBTSEFQRVM6CiAgICAgICAgICAgIGRhdGE9Zml4dHVyZShtLG4sVHJ1ZSk7YmVmb3JlPWcuQ09VTlRTWydjdXN0b20nXQogICAgICAgICAgICB3aXRoIGN0eCgpOgogICAgICAgICAgICAgICAgdG9yY2guY3VkYS5zeW5jaHJvbml6ZSgpO3N0YXJ0PXRpbWUucGVyZl9jb3VudGVyKCk7YWN0dWFsPWNhbGwoZGF0YSkKICAgICAgICAgICAgICAgIHRvcmNoLmN1ZGEuc3luY2hyb25pemUoKTtjb2xkPXRpbWUucGVyZl9jb3VudGVyKCktc3RhcnQKICAgICAgICAgICAgZXhwZWN0ZWQ9Y2FsbChkYXRhKTtudW1lcmljPWVuZ2luZWVyaW5nX2Nsb3NlKGFjdHVhbCxleHBlY3RlZCkKICAgICAgICAgICAgZGVsIGFjdHVhbCxleHBlY3RlZAogICAgICAgICAgICBpZiBhcmdzLm1ldGhvZD09J2NhbmRpZGF0ZScgYW5kIGcuQ09VTlRTWydjdXN0b20nXSE9YmVmb3JlKzE6cmFpc2UgUnVudGltZUVycm9yKCdDb2xkIGNhbmRpZGF0ZSBpbmFjdGl2ZScpCiAgICAgICAgICAgIHBhaXJzPVtdCiAgICAgICAgICAgIHdpdGggY3R4KCk6CiAgICAgICAgICAgICAgICBmb3IgXyBpbiByYW5nZSgzKTpjYWxsKGRhdGEpCiAgICAgICAgICAgICAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKCkKICAgICAgICAgICAgICAgIGZvciBfIGluIHJhbmdlKDEyKToKICAgICAgICAgICAgICAgICAgICBzdGFydD10b3JjaC5jdWRhLkV2ZW50KGVuYWJsZV90aW1pbmc9VHJ1ZSk7c3RvcD10b3JjaC5jdWRhLkV2ZW50KGVuYWJsZV90aW1pbmc9VHJ1ZSkKICAgICAgICAgICAgICAgICAgICBzdGFydC5yZWNvcmQoKTtjYWxsKGRhdGEpO3N0b3AucmVjb3JkKCk7cGFpcnMuYXBwZW5kKChzdGFydCxzdG9wKSkKICAgICAgICAgICAgICAgIHRvcmNoLmN1ZGEuc3luY2hyb25pemUoKQogICAgICAgICAgICBzYW1wbGVzPVthLmVsYXBzZWRfdGltZShiKSBmb3IgYSxiIGluIHBhaXJzXQogICAgICAgICAgICBpZiBhcmdzLm1ldGhvZD09J2NhbmRpZGF0ZScgYW5kIGcuQ09VTlRTWydjdXN0b20nXSE9YmVmb3JlKzE2OnJhaXNlIFJ1bnRpbWVFcnJvcignV2FybSBjYW5kaWRhdGUgaW5hY3RpdmUnKQogICAgICAgICAgICByb3c9eydNJzptLCdOJzpuLCdLJzoxNTM2LCdiaWFzJzpUcnVlLCdjb2xkX2ZpcnN0X3dyYXBwZXJfd2FsbF9zJzpjb2xkLAogICAgICAgICAgICAgICAgICd3YXJtX2ZvcndhcmRfY3VkYV9ldmVudF9tcyc6c2FtcGxlcywnbnVtZXJpY3MnOm51bWVyaWMsJ2NvdW50cyc6ZGljdChnLkNPVU5UUyl9CiAgICAgICAgICAgIHJlcG9ydFsncm93cyddLmFwcGVuZChyb3cpO291dC53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVwb3J0LGluZGVudD0yLGFsbG93X25hbj1GYWxzZSkrJ1xuJykKICAgICAgICBpZiBsZW4ocmVwb3J0Wydyb3dzJ10pIT02OnJhaXNlIFJ1bnRpbWVFcnJvcignSW5jb21wbGV0ZSBmaXh0dXJlIHNldCcpCiAgICAgICAgcmVwb3J0LnVwZGF0ZShzdGF0dXM9J1BBU1NFRF9NSUNST0JFTkNIX09OTFknLGdwdV9leGVjdXRlZD1UcnVlKQogICAgZXhjZXB0IEJhc2VFeGNlcHRpb24gYXMgZTpyZXBvcnQudXBkYXRlKGVycm9yPXJlcHIoZSksdHJhY2ViYWNrPXRyYWNlYmFjay5mb3JtYXRfZXhjKCkpCiAgICBvdXQud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlcG9ydCxpbmRlbnQ9MixhbGxvd19uYW49RmFsc2UpKydcbicpCiAgICBwcmludChqc29uLmR1bXBzKHsnc3RhdHVzJzpyZXBvcnRbJ3N0YXR1cyddLCdtZXRob2QnOmFyZ3MubWV0aG9kLCdyb3dzJzpsZW4ocmVwb3J0Wydyb3dzJ10pfSksZmx1c2g9VHJ1ZSkKICAgIHJhaXNlIFN5c3RlbUV4aXQocmVwb3J0WydzdGF0dXMnXSE9J1BBU1NFRF9NSUNST0JFTkNIX09OTFknKQppZiBfX25hbWVfXz09J19fbWFpbl9fJzptYWluKCkK', 'driver.py': 'IiIiQm91bmRlZCBuYXRpdmUvY2FuZGlkYXRlIG9yaWdpbmFsIE5GNCBmb3J3YXJkIE1JQ1JPQkVOQ0hfT05MWSBvcmNoZXN0cmF0aW9uLgoKTmV2ZXIgaW5zdGFsbHMgaW50byBTT1VQX1JFUE8gb3IgY2FsbHMgYSB0cmFpbmluZyBoYXJuZXNzLiBFeGlzdGluZyBzb3VyY2UvcnVudGltZSwKbG9jaywgcGlkZmQvc3VicmVhcGVyLCBxdWFyYW50aW5lIGFuZCBpZGxlLWNoZWNrIGhlbHBlcnMgYXJlIHByZXNlcnZlZCB1bmNoYW5nZWQuClRoZSBkZXBsb3llZCBzZXJ2aWNlJ3MgbGltaXRzIHJlbWFpbiBhdXRob3JpdGF0aXZlOyBlYWNoIEdQVSBjaGlsZCBnZXRzIDw9ODAgcy4KIiIiCmltcG9ydCBmY250bCwgaGFzaGxpYiwganNvbiwgb3MsIHN5cywgdGltZSwgdXVpZApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHByb2Nlc3NfZ3VhcmQKaW1wb3J0IG93bmVkX3Byb2Nlc3NfdHJlZSBhcyB0cmVlCmZyb20gcmV2aWV3ZWRfc3VwcG9ydCBpbXBvcnQgdmVyaWZ5X29mZmljaWFsLCBpZGxlX2NoZWNrLCBFeHBvcnRlciwgZXhwb3J0X3NtYWxsLCByZWFkLCB3cml0ZSwgc2hhCgpMT0NLID0gUGF0aCgnL3RtcC9zb3VwLWdwdTAubG9jaycpCk1BUktFUiA9IFBhdGgoJy90bXAvc291cC1ncHUwLmxvY2suYmxvY2tlZCcpCgpkZWYgbWFpbigpOgogICAgaGVyZSA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQKICAgIGlmIG5vdCBvcy5lbnZpcm9uLmdldCgnU09VUF9SRVBPJykgb3Igbm90IG9zLmVudmlyb24uZ2V0KCdTT1VQX1JFU1VMVFMnKToKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KCdSZXF1aXJlcyBkZXBsb3llZCBTT1VQX1JFUE8vU09VUF9SRVNVTFRTLCBubyBtYW51YWwgZmFsbGJhY2snKQogICAgcmVwbywgcmVzdWx0cyA9IFBhdGgob3MuZW52aXJvblsnU09VUF9SRVBPJ10pLmFic29sdXRlKCksIFBhdGgob3MuZW52aXJvblsnU09VUF9SRVNVTFRTJ10pLmFic29sdXRlKCkKICAgIGlmIHJlcG8uaXNfc3ltbGluaygpIG9yIHJlc3VsdHMuaXNfc3ltbGluaygpIG9yIHJlcG8gPT0gcmVzdWx0czoKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KCdQcmVwYXJlZCBjaGVja291dC9yZXN1bHRzIG11c3QgYmUgc2VwYXJhdGUgcmVhbCBkaXJlY3RvcmllcycpCiAgICBhc3NlcnQgUGF0aCgnL2thZ2dsZS93b3JraW5nJykuaXNfZGlyKCkKICAgIHJlc3VsdHMubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICBpZGVudGl0eSA9ICduZjQtbW1hLW1pY3JvYmVuY2gtb25seS0nICsgdXVpZC51dWlkNCgpLmhleFs6MTJdCiAgICBydW4sIGV4cG9ydCA9IFBhdGgoJy9rYWdnbGUvd29ya2luZycpL2lkZW50aXR5LCByZXN1bHRzL2lkZW50aXR5CiAgICBydW4ubWtkaXIoZXhpc3Rfb2s9RmFsc2UpOyBleHBvcnQubWtkaXIoZXhpc3Rfb2s9RmFsc2UpCiAgICBzdGF0dXMgPSBkaWN0KHN0YXR1cz0nUFJFUEFSSU5HX01JQ1JPQkVOQ0hfT05MWScsc2NvcGU9J01JQ1JPQkVOQ0hfT05MWScsCiAgICAgICAgICAgICAgICAgIHRyYWluaW5nX2V4ZWN1dGVkPUZhbHNlLGNvbXBsZXRlX2ZpeGVkX3Rhc2s9RmFsc2Usc2NvcmU9Tm9uZSxhZG1pc3Npb249J1VOS05PV04nLAogICAgICAgICAgICAgICAgICBwZXJfZ3B1X2NoaWxkX3dhbGxfY2FwX3M9ODAsY2FwX2J5cGFzcz1GYWxzZSwKICAgICAgICAgICAgICAgICAgY2FuZGlkYXRlX3NvdXJjZV9zaGEyNTY9cmVhZChoZXJlLydzb3VyY2VfbWFuaWZlc3QuanNvbicpLAogICAgICAgICAgICAgICAgICBtZXRob2RzX2luX2V4ZWN1dGlvbl9vcmRlcj1bJ25hdGl2ZScsJ2NhbmRpZGF0ZSddLHByb2Nlc3Nlcz17fSkKICAgIHdyaXRlKGV4cG9ydC8nc3RhdHVzLmpzb24nLHN0YXR1cykKICAgIGxvY2tmZCA9IE5vbmU7IHN0YXJ0ZWQgPSB0aW1lLm1vbm90b25pYygpCiAgICB0cnk6CiAgICAgICAgZm9yIG5hbWUsZGlnZXN0IGluIHJlYWQoaGVyZS8ncGF5bG9hZF9tYW5pZmVzdC5qc29uJykuaXRlbXMoKToKICAgICAgICAgICAgcCA9IGhlcmUvbmFtZQogICAgICAgICAgICBhc3NlcnQgbm90IHAuaXNfc3ltbGluaygpIGFuZCBwLmlzX2ZpbGUoKSBhbmQgc2hhKHApPT1kaWdlc3QsICdQYXlsb2FkIGNoYW5nZWQ6ICcrbmFtZQogICAgICAgIGlmIE1BUktFUi5leGlzdHMoKTogcmFpc2UgUnVudGltZUVycm9yKCdHUFUgcXVhcmFudGluZSByZXF1aXJlcyBvcGVyYXRvciByZWNvbmNpbGlhdGlvbicpCiAgICAgICAgbG9ja2ZkID0gb3Mub3BlbihMT0NLLG9zLk9fQ1JFQVR8b3MuT19SRFdSfG9zLk9fTk9GT0xMT1csMG82MDApCiAgICAgICAgZmNudGwuZmxvY2sobG9ja2ZkLGZjbnRsLkxPQ0tfRVh8ZmNudGwuTE9DS19OQikKICAgICAgICBpZiBNQVJLRVIuZXhpc3RzKCk6IHJhaXNlIFJ1bnRpbWVFcnJvcignR1BVIHF1YXJhbnRpbmUgYXBwZWFyZWQgd2hpbGUgbG9ja2luZycpCiAgICAgICAgdHJlZS5wcmVwYXJlKCk7IHRyZWUuYmVnaW5fcXVhcmFudGluZShNQVJLRVIpCiAgICAgICAgb2ZmaWNpYWwgPSByZWFkKGhlcmUvJ29mZmljaWFsX21hbmlmZXN0Lmpzb24nKQogICAgICAgIHZlcmlmeV9vZmZpY2lhbChyZXBvLG9mZmljaWFsKQogICAgICAgIGVudiA9IGRpY3Qob3MuZW52aXJvbixIRl9IVUJfT0ZGTElORT0nMScsSEZfREFUQVNFVFNfT0ZGTElORT0nMScsVFJBTlNGT1JNRVJTX09GRkxJTkU9JzEnLAogICAgICAgICAgICAgICAgICAgUFlUSE9ORE9OVFdSSVRFQllURUNPREU9JzEnLFBZVEhPTlVOQlVGRkVSRUQ9JzEnKQogICAgICAgIGVudi5wb3AoJ1NPVVBfUk9UQVJZX0NPREVHRU5fRElSJyxOb25lKQogICAgICAgIHdpdGggRXhwb3J0ZXIocnVuLGV4cG9ydCk6CiAgICAgICAgICAgIHByb2JlID0gcHJvY2Vzc19ndWFyZC5ydW4oW3N5cy5leGVjdXRhYmxlLCctQicsc3RyKGhlcmUvJ3J1bnRpbWVfcHJvYmUucHknKSxzdHIocnVuLydydW50aW1lX3Byb2JlLmpzb24nKV0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY3dkPXJlcG8sZW52PWVudixsb2c9cnVuLydydW50aW1lX3Byb2JlLmxvZycsdGltZW91dD00MCxsb2NrX2ZkPWxvY2tmZCkKICAgICAgICAgICAgc3RhdHVzWydwcm9jZXNzZXMnXVsncnVudGltZV9wcm9iZSddID0gcHJvYmUKICAgICAgICAgICAgaWYgcHJvYmVbJ2V4aXRfY29kZSddIG9yIG5vdCByZWFkKHJ1bi8ncnVudGltZV9wcm9iZS5qc29uJykuZ2V0KCdwYXNzZWQnKToKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignRXhhY3QgZGVwbG95ZWQgcnVudGltZSBnYXRlIGZhaWxlZDsgbm8gYmVuY2htYXJrJykKICAgICAgICAgICAgZGVmIGlkbGUobGFiZWwpOgogICAgICAgICAgICAgICAgZm9sZGVyPXJ1bi8oJ2lkbGVfJytsYWJlbCk7Zm9sZGVyLm1rZGlyKGV4aXN0X29rPUZhbHNlKQogICAgICAgICAgICAgICAgcmVzdWx0PWlkbGVfY2hlY2soZW52LHJlcG8sZm9sZGVyLGxvY2tmZCkKICAgICAgICAgICAgICAgIHN0YXR1c1sncHJvY2Vzc2VzJ11bJ2lkbGVfJytsYWJlbF09cmVzdWx0CiAgICAgICAgICAgICAgICBpZiByZXN1bHRbJ2V4aXRfY29kZSddOnJhaXNlIFJ1bnRpbWVFcnJvcignSWRsZSBnYXRlIGZhaWxlZDogJytsYWJlbCkKICAgICAgICAgICAgaWRsZSgnYmVmb3JlX3ByZWZsaWdodCcpCiAgICAgICAgICAgICMgRXhpc3Rpbmcgb3JpZ2luYWwgbnVtZXJpY2FsL1BUWCBnYXRlLCBpc29sYXRlZCBmcm9tIGV2ZXJ5IG1lYXN1cmVkIGNhY2hlLgogICAgICAgICAgICBwcmVlbnYgPSBkaWN0KGVudixUUklUT05fQ0FDSEVfRElSPXN0cihydW4vJ3ByZWZsaWdodF90cml0b24nKSwKICAgICAgICAgICAgICAgICAgICAgICAgICBUT1JDSElORFVDVE9SX0NBQ0hFX0RJUj1zdHIocnVuLydwcmVmbGlnaHRfaW5kdWN0b3InKSkKICAgICAgICAgICAgcHJlID0gcHJvY2Vzc19ndWFyZC5ydW4oW3N5cy5leGVjdXRhYmxlLCctQicsJy1tJywnc29sdXRpb24ud2ViX3ByZWZsaWdodCcsJy0tb3V0JyxzdHIocnVuLydwcmVmbGlnaHQuanNvbicpXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY3dkPWhlcmUsZW52PXByZWVudixsb2c9cnVuLydwcmVmbGlnaHQubG9nJyx0aW1lb3V0PTgwLGxvY2tfZmQ9bG9ja2ZkKQogICAgICAgICAgICBzdGF0dXNbJ3Byb2Nlc3NlcyddWydwcmVmbGlnaHQnXSA9IHByZQogICAgICAgICAgICBpZiBwcmVbJ2V4aXRfY29kZSddIG9yIHJlYWQocnVuLydwcmVmbGlnaHQuanNvbicpLmdldCgnc3RhdHVzJykhPSdQQVNTRURfUFJFRkxJR0hUX09OTFknOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdPcmlnaW5hbCBORjQgTU1BIG51bWVyaWNhbC9QVFggZ2F0ZSBmYWlsZWQ7IG5vIGJlbmNobWFyaycpCiAgICAgICAgICAgIHN0YXR1c1snc3RhdHVzJ109J1JVTk5JTkdfTUlDUk9CRU5DSF9PTkxZJzt3cml0ZShleHBvcnQvJ3N0YXR1cy5qc29uJyxzdGF0dXMpCiAgICAgICAgICAgIGlkbGUoJ3ByZWZsaWdodCcpCiAgICAgICAgICAgIGZvciBtZXRob2QgaW4gc3RhdHVzWydtZXRob2RzX2luX2V4ZWN1dGlvbl9vcmRlciddOgogICAgICAgICAgICAgICAgY2FjaGU9cnVuLyd0cml0b25fY2FjaGUnL21ldGhvZDtjYWNoZS5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9RmFsc2UpCiAgICAgICAgICAgICAgICBjaGlsZGVudj1kaWN0KGVudixUUklUT05fQ0FDSEVfRElSPXN0cihjYWNoZSksVE9SQ0hJTkRVQ1RPUl9DQUNIRV9ESVI9c3RyKHJ1bi8naW5kdWN0b3JfY2FjaGUnL21ldGhvZCkpCiAgICAgICAgICAgICAgICBwcm9jZXNzPXByb2Nlc3NfZ3VhcmQucnVuKFtzeXMuZXhlY3V0YWJsZSwnLUInLHN0cihoZXJlLyd3b3JrZXIucHknKSwnLS1tZXRob2QnLG1ldGhvZCwnLS1vdXQnLHN0cihydW4vKG1ldGhvZCsnLmpzb24nKSldLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjd2Q9aGVyZSxlbnY9Y2hpbGRlbnYsbG9nPXJ1bi8obWV0aG9kKycubG9nJyksdGltZW91dD04MCxsb2NrX2ZkPWxvY2tmZCkKICAgICAgICAgICAgICAgIHN0YXR1c1sncHJvY2Vzc2VzJ11bbWV0aG9kXT1wcm9jZXNzCiAgICAgICAgICAgICAgICBpZiBwcm9jZXNzWydleGl0X2NvZGUnXSBvciByZWFkKHJ1bi8obWV0aG9kKycuanNvbicpKS5nZXQoJ3N0YXR1cycpIT0nUEFTU0VEX01JQ1JPQkVOQ0hfT05MWSc6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdJbmNvbXBsZXRlL2ZhaWxlZC9jZW5zb3JlZCBtaWNyb2JlbmNobWFyazogJyttZXRob2QpCiAgICAgICAgICAgICAgICBpZGxlKG1ldGhvZCkKICAgICAgICAgICAgICAgIHdyaXRlKGV4cG9ydC8nc3RhdHVzLmpzb24nLHN0YXR1cykKICAgICAgICAgICAgdmVyaWZ5X29mZmljaWFsKHJlcG8sb2ZmaWNpYWwpCiAgICAgICAgICAgIHJlcG9ydHM9e21ldGhvZDpyZWFkKHJ1bi8obWV0aG9kKycuanNvbicpKSBmb3IgbWV0aG9kIGluIHN0YXR1c1snbWV0aG9kc19pbl9leGVjdXRpb25fb3JkZXInXX0KICAgICAgICAgICAgYXNzZXJ0IGFsbChsZW4oclsncm93cyddKT09NiBhbmQgclsndHJhaW5pbmdfZXhlY3V0ZWQnXSBpcyBGYWxzZSBhbmQgclsnY29tcGxldGVfZml4ZWRfdGFzayddIGlzIEZhbHNlIGZvciByIGluIHJlcG9ydHMudmFsdWVzKCkpCiAgICAgICAgICAgIHdyaXRlKHJ1bi8nY29tcGFyaXNvbi5qc29uJyxkaWN0KHNjb3BlPSdNSUNST0JFTkNIX09OTFknLHJlcG9ydHM9cmVwb3J0cywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzY29yZT1Ob25lLGFkbWlzc2lvbj0nVU5LTk9XTicsY29tcGxldGVfZml4ZWRfdGFzaz1GYWxzZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBub3RlPSdSYXcgZmlyc3QtY2FsbCB3YWxsIGFuZCB3YXJtIENVREEtZXZlbnQgc2FtcGxlcywgbm90IHB1cmUgY29tcGlsZSBjb3N0IG9yIHRhc2sgc3BlZWR1cDsgbm8gcmF0aW9zIG9yIG93bmVyc2hpcC1zaGFyZSBpbmZlcmVuY2UuJykpCiAgICAgICAgICAgIHN0YXR1c1snc3RhdHVzJ109J1BBU1NFRF9NSUNST0JFTkNIX09OTFknCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgc3RhdHVzLnVwZGF0ZShzdGF0dXM9J0JMT0NLRURfRkFJTEVEX09SX0NFTlNPUkVEX01JQ1JPQkVOQ0hfT05MWScsZXJyb3I9cmVwcihleGMpKQogICAgICAgIGlmIGxvY2tmZCBpcyBub3QgTm9uZTp0cmVlLnF1YXJhbnRpbmUoJ05GNCBtaWNyb2JlbmNobWFyayBmYWlsZWQ6ICcrcmVwcihleGMpKQogICAgZmluYWxseToKICAgICAgICBzdGF0dXNbJ29yY2hlc3RyYXRvcl93YWxsX3MnXT10aW1lLm1vbm90b25pYygpLXN0YXJ0ZWQKICAgICAgICBleHBvcnRfc21hbGwocnVuLGV4cG9ydCk7d3JpdGUoZXhwb3J0LydzdGF0dXMuanNvbicsc3RhdHVzKQogICAgICAgIGlmIGxvY2tmZCBpcyBub3QgTm9uZToKICAgICAgICAgICAgdHJlZS5maW5pc2hfcXVhcmFudGluZSgpO29zLmNsb3NlKGxvY2tmZCkKICAgIHByaW50KGpzb24uZHVtcHMoeydzY29wZSc6J01JQ1JPQkVOQ0hfT05MWScsJ3N0YXR1cyc6c3RhdHVzWydzdGF0dXMnXSwnZXhwb3J0JzpzdHIoZXhwb3J0KSwKICAgICAgICAgICAgICAgICAgICAgICdzY29yZSc6Tm9uZSwnYWRtaXNzaW9uJzonVU5LTk9XTid9KSxmbHVzaD1UcnVlKQogICAgcmFpc2UgU3lzdGVtRXhpdCgwIGlmIHN0YXR1c1snc3RhdHVzJ109PSdQQVNTRURfTUlDUk9CRU5DSF9PTkxZJyBlbHNlIDEpCgppZiBfX25hbWVfXz09J19fbWFpbl9fJzptYWluKCkK', 'source_manifest.json': 'ewogICJzb2x1dGlvbi9fX2luaXRfXy5weSI6ICJiNDkzZjE3ZDgzMDhmYzVjYmUyZGIwN2Q2ZWZjYjE1OGIxMDkwMzViMzI0N2UxNGU3NzZlMjU3YzY1ZDQzYTQxIiwKICAic29sdXRpb24vZml4dHVyZXMucHkiOiAiYjZmZTA2NmQzMTNjMjkwMTY3NzVlZjYwYzg0YWZhNjM4MTYzMDg1NjllNDY2ZTk0MTBhMjNhZWRmMTNmM2M1MCIsCiAgInNvbHV0aW9uL2d1YXJkZWRfZm9yd2FyZC5weSI6ICJlYzI5MjQwMmFmOGE0ZTg1NWE1MDVkNjQ3MWU3MTNhNzEzMDc5MzMwMjVmYzk3YjRjOGQyMmU0ZWRhMGVjZDFiIiwKICAic29sdXRpb24va2VybmVsLnB5IjogImIwMzc5NWYxZTRmZWFhOTliNzhkZWUzZDFiOGY4YzFlYWNhMzc5NDNhODJlZGMzZWYxNDAwZTFjYTRjYzFjZjgiLAogICJzb2x1dGlvbi9uYXRpdmVfc291cmNlX21hbmlmZXN0Lmpzb24iOiAiOTA1Yjg4MDYzOTQxYzAzOWFlMzBmMTQ3YzUyNDgzMmFhZDc3NjZlOTllYjFjNDUxOTk4YmFjNDM0MTdiNmFjNyIsCiAgInNvbHV0aW9uL25mNF9mb3JtYXQucHkiOiAiMjExODlmOTdhNzQ1MmE5NjA3ZTI4OTFiZTA0MDJjZGI0NTViNzIxOTE5Y2Q5MWRhZWRkYTE2YzFkN2M4YjBlNCIsCiAgInNvbHV0aW9uL29wZXJhbmRfcHJvYmUucHkiOiAiNTQ2YTYyYTZkMGJiOGJmN2Q5NWJlZTQzNGRhNTVkOGUxMTExY2U4ZTQ1YmZjZDc4MmZiNjRmODA4NzM4ODUyOCIsCiAgInNvbHV0aW9uL3dlYl9wcmVmbGlnaHQucHkiOiAiOTlhZGU2ZjBiZjI2ZWZlZWIwM2E5MmUzNTg1MzY3MDY0NGY0OTI1NmJjODlhM2I1NDMwOWIwMzZiYzVmYTM4YSIKfQo=', 'payload_manifest.json': 'ewogICJzb2x1dGlvbi9fX2luaXRfXy5weSI6ICJiNDkzZjE3ZDgzMDhmYzVjYmUyZGIwN2Q2ZWZjYjE1OGIxMDkwMzViMzI0N2UxNGU3NzZlMjU3YzY1ZDQzYTQxIiwKICAic29sdXRpb24vZml4dHVyZXMucHkiOiAiYjZmZTA2NmQzMTNjMjkwMTY3NzVlZjYwYzg0YWZhNjM4MTYzMDg1NjllNDY2ZTk0MTBhMjNhZWRmMTNmM2M1MCIsCiAgInNvbHV0aW9uL2d1YXJkZWRfZm9yd2FyZC5weSI6ICJlYzI5MjQwMmFmOGE0ZTg1NWE1MDVkNjQ3MWU3MTNhNzEzMDc5MzMwMjVmYzk3YjRjOGQyMmU0ZWRhMGVjZDFiIiwKICAic29sdXRpb24va2VybmVsLnB5IjogImIwMzc5NWYxZTRmZWFhOTliNzhkZWUzZDFiOGY4YzFlYWNhMzc5NDNhODJlZGMzZWYxNDAwZTFjYTRjYzFjZjgiLAogICJzb2x1dGlvbi9uYXRpdmVfc291cmNlX21hbmlmZXN0Lmpzb24iOiAiOTA1Yjg4MDYzOTQxYzAzOWFlMzBmMTQ3YzUyNDgzMmFhZDc3NjZlOTllYjFjNDUxOTk4YmFjNDM0MTdiNmFjNyIsCiAgInNvbHV0aW9uL25mNF9mb3JtYXQucHkiOiAiMjExODlmOTdhNzQ1MmE5NjA3ZTI4OTFiZTA0MDJjZGI0NTViNzIxOTE5Y2Q5MWRhZWRkYTE2YzFkN2M4YjBlNCIsCiAgInNvbHV0aW9uL29wZXJhbmRfcHJvYmUucHkiOiAiNTQ2YTYyYTZkMGJiOGJmN2Q5NWJlZTQzNGRhNTVkOGUxMTExY2U4ZTQ1YmZjZDc4MmZiNjRmODA4NzM4ODUyOCIsCiAgInNvbHV0aW9uL3dlYl9wcmVmbGlnaHQucHkiOiAiOTlhZGU2ZjBiZjI2ZWZlZWIwM2E5MmUzNTg1MzY3MDY0NGY0OTI1NmJjODlhM2I1NDMwOWIwMzZiYzVmYTM4YSIsCiAgInByb2Nlc3NfZ3VhcmQucHkiOiAiODI0NjU5NDNhM2E5YWYzODMxYTUzOTlhM2E1NWQwOWFjMjkwYmE3MzY1YzRlODBkNmJjNWU3OWZhYjYyYTZmZiIsCiAgIm93bmVkX3Byb2Nlc3NfdHJlZS5weSI6ICI2OTNkZGU3MDhkYTVjMDYwNTNhZjIzOTQ2Mzk1MjExNWVmNWU4ODY1ZjUwYmNiYzUxYTIwMzgyMzdjYTU3ZDY5IiwKICAicnVudGltZV9wcm9iZS5weSI6ICJlZmUxYjdjZjNmM2NmMWUzN2JiYzE2OWE4NDY4ODU5YzUxMjdlYzMyODFhMzdlMzkwNjg3MDNiMmViM2MxNDgwIiwKICAib2ZmaWNpYWxfbWFuaWZlc3QuanNvbiI6ICJkMDEzZTE2MzM5NDBjMjdkNzA4ZTI0NzRhODI4MDk5MDAwMGUzMzAxOGU4ODllZDE0NjU1Yzg1NjMyOWZmMWQ4IiwKICAicmV2aWV3ZWRfc3VwcG9ydC5weSI6ICI3YzVkNjAyZDA3MTM4M2VmMDdmZGRkOTk2NzBlZDFjY2UzMTQxZmVhYmQ1YWY4Mzk0ZjA1M2M2YjRlMDcxMjJjIiwKICAid29ya2VyLnB5IjogImMyN2IxNjQyMTZmNmNlYjcxZDAxNDVmYWY5NjVhNzI4Nzk3NjhlMzljOGFmYzNlMGRkNmYxMmQ2MmJlZTFiYTciLAogICJkcml2ZXIucHkiOiAiNjhiYzI0Nzk5YmY2N2IyZDI3ZWI2MTFmNDBlZWU4YWExNWU5M2QyMWU5NzRlNjdmZGJmOGM4MmU0OTBjMDhjMCIsCiAgInNvdXJjZV9tYW5pZmVzdC5qc29uIjogIjc3ZTJkNTM0MDcyYjI2ZjgwNDViY2ZiYjFlYzEzMTg3MDYzNGJkZjA5OThiNzg0MTE4Y2IxYzcwOGM3YzQ1ZjUiCn0K'}
PAYLOAD_SHA256 = {'solution/__init__.py': 'b493f17d8308fc5cbe2db07d6efcb158b109035b3247e14e776e257c65d43a41', 'solution/fixtures.py': 'b6fe066d313c29016775ef60c84afa63816308569e466e9410a23aedf13f3c50', 'solution/guarded_forward.py': 'ec292402af8a4e855a505d6471e713a71307933025fc97b4c8d22e4eda0ecd1b', 'solution/kernel.py': 'b03795f1e4feaa99b78dee3d1b8f8c1eaca37943a82edc3ef1400e1ca4cc1cf8', 'solution/native_source_manifest.json': '905b88063941c039ae30f147c524832aad7766e99eb1c451998bac43417b6ac7', 'solution/nf4_format.py': '21189f97a7452a9607e2891be0402cdb455b721919cd91daedda16c1d7c8b0e4', 'solution/operand_probe.py': '546a62a6d0bb8bf7d95bee434da55d8e1111ce8e45bfcd782fb64f8087388528', 'solution/web_preflight.py': '99ade6f0bf26efeeb03a92e35853670644f49256bc89a3b54309b036bc5fa38a', 'process_guard.py': '82465943a3a9af3831a5399a3a55d09ac290ba7365c4e80d6bc5e79fab62a6ff', 'owned_process_tree.py': '693dde708da5c06053af239463952115ef5e8865f50bcbc51a2038237ca57d69', 'runtime_probe.py': 'efe1b7cf3f3cf1e37bbc169a8468859c5127ec3281a37e39068703b2eb3c1480', 'official_manifest.json': 'd013e1633940c27d708e2474a8280990000e33018e889ed14655c856329ff1d8', 'reviewed_support.py': '7c5d602d071383ef07fddd99670ed1cce3141feabd5af8394f053c6b4e07122c', 'worker.py': 'c27b164216f6ceb71d0145faf965a72879768e39c8afc3e0dd6f12d62bee1ba7', 'driver.py': '68bc24799bf67b2d27eb611f40eee8aa15e93d21e974e67fdbf8c82e490c08c0', 'source_manifest.json': '77e2d534072b26f8045bcfbb1ec131870634bdf0998b784118cb1c708c7c45f5', 'payload_manifest.json': 'ed1f6dfc86f2ca503bca855b724f4116e892739b54390db5a5ce0fd4f44fd06e'}
STAGED=Path(tempfile.mkdtemp(prefix='nf4-mma-microbench-',dir='/kaggle/working'))
for name,encoded in PAYLOAD.items():
    parts=Path(name).parts
    assert not Path(name).is_absolute() and '..' not in parts
    raw=base64.b64decode(encoded,validate=True)
    assert hashlib.sha256(raw).hexdigest()==PAYLOAD_SHA256[name]
    target=STAGED/name;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(raw)
print('Prepared MICROBENCH_ONLY payload:',STAGED)


In [ ]:
import subprocess,sys,signal
process=subprocess.Popen([sys.executable,'-B',str(STAGED/'driver.py')],start_new_session=True)
try:
    return_code=process.wait()
except BaseException:
    if process.poll() is None:process.send_signal(signal.SIGTERM)
    while process.poll() is None:
        try:process.wait()
        except KeyboardInterrupt:pass
    raise
if return_code:raise RuntimeError('MICROBENCH_ONLY failed or censored; inspect exported status. No automatic retry.')
